# Klinik Chong RAG Pipeline — Run-All Version
This notebook loads the existing ChromaDB index and implements two `ask_info` branches:

1. **Slot availability:** Query Compiler → Slot Intent Gate → SQL Parameter Extractor → SQLite only.
2. **Static clinic information:** Query Compiler → Main Section Analyzer → Subsection Analyzer → ChromaDB → Context Compiler → Response Generator.

The ingestion code is preserved for reproducibility but is skipped by default because the persistent ChromaDB already exists.


In [ ]:
# Install all packages required by the complete RAG notebook.
%pip install -q "Pillow==11.3.0" pdfplumber pandas numpy matplotlib sentence-transformers chromadb transformers datasets accelerate sentencepiece evaluate sacrebleu langchain-core langchain-openai


### Load Libraries
- Imports PDF processing, data handling, embedding, vector database, evaluation and LangChain dependencies used throughout the notebook.


In [ ]:
# Load all libraries used throughout the notebook.
import json
import os
import re
import sqlite3
from difflib import SequenceMatcher
from datetime import date, datetime, time as datetime_time, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

from google.colab import drive, userdata
import chromadb
import numpy as np
import pandas as pd
import pdfplumber
import PIL
import torch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import SentenceTransformer


## 1. Ingestion
- Uses the persistent ChromaDB by default so **Run All** does not repeat PDF ingestion.
- Set `REBUILD_INDEX = True` only when the handbook content or chunking logic changes.
- The original ingestion, structure extraction, metadata and embedding cells remain available for reproducibility.


### Define Source PDF Path
- Points to the Klinik Chong patient handbook stored in Google Drive.


In [ ]:
drive.mount("/content/drive")

FYP2_ROOT = Path("/content/drive/MyDrive/FYP2")
DATABASE_DIR = FYP2_ROOT / "klinik_chong_database"
DB_PATH = DATABASE_DIR / "klinik_chong.db"
PDF_PATH = DATABASE_DIR / "Klinik Chong Buku Panduan Pesakit.pdf"
CHROMA_DB_PATH = DATABASE_DIR / "chroma_db"

# Existing ChromaDB is the normal Run-All path.
REBUILD_INDEX = False

# Expensive development evaluations are disabled during normal Run All.
# Change a flag to True only when a fresh evaluation is intentionally required.
RUN_LOCAL_RETRIEVAL_EVALUATION = False
RUN_OPENAI_API_TESTS = False
RUN_SQL_LLM_EVALUATION = False

# Retrieved handbook text below this score is treated as unsupported context.
MIN_RAG_SIMILARITY = 0.50

if REBUILD_INDEX and not PDF_PATH.exists():
    raise FileNotFoundError(f"Handbook PDF not found: {PDF_PATH}")
if not CHROMA_DB_PATH.exists():
    raise FileNotFoundError(f"ChromaDB folder not found: {CHROMA_DB_PATH}")

print("Run-All mode:", "REBUILD INDEX" if REBUILD_INDEX else "USE EXISTING CHROMADB")
print("OpenAI API tests:", "ENABLED" if RUN_OPENAI_API_TESTS else "SKIPPED")
print("SQL LLM evaluation:", "ENABLED" if RUN_SQL_LLM_EVALUATION else "SKIPPED")
print("Handbook PDF:", PDF_PATH)
print("ChromaDB:", CHROMA_DB_PATH)


### Inspect Source PDF
- Opens the source PDF and checks page content before cleaning.


In [ ]:
if REBUILD_INDEX:
    # check page
    with pdfplumber.open(PDF_PATH) as pdf:

        print("Total PDF pages:", len(pdf.pages))

        sample_page = pdf.pages[4]
        sample_text = sample_page.extract_text()

    print("\n===== RAW PAGE 5 =====\n")
    print(sample_text)


### Define Text Cleaning
- Removes repeated headers, footers and other noise from extracted page text.


In [ ]:
if REBUILD_INDEX:
    # clean noise
    REPEATED_HEADERS = {
        "Klinik Chong",
        "Buku Panduan Pesakit"
    }


    def clean_pdf_page_text(text):

        if text is None:
            return ""

        cleaned_lines = []

        for raw_line in text.splitlines():

            # Remove extra spaces inside each line
            line = re.sub(
                r"\s+",
                " ",
                raw_line
            ).strip()

            # Remove empty lines
            if not line:
                continue

            # Remove repeated document headers
            if line in REPEATED_HEADERS:
                continue

            # Remove standalone footer page numbers
            if re.fullmatch(r"\d+", line):
                continue

            cleaned_lines.append(line)

        return "\n".join(cleaned_lines)


### Define Pages to Skip
- Lists handbook pages that should not enter the knowledge base.


In [ ]:
if REBUILD_INDEX:
    SKIP_PDF_PAGES = {
        1,  # Cover page
        3,  # Table of contents
        4   # Table of contents
    }


    cleaned_pages = []


    with pdfplumber.open(PDF_PATH) as pdf:

        total_pdf_pages = len(pdf.pages)

        for pdf_page_number, page in enumerate(
            pdf.pages,
            start=1
        ):

            if pdf_page_number in SKIP_PDF_PAGES:
                continue

            raw_text = page.extract_text() or ""

            cleaned_text = clean_pdf_page_text(
                raw_text
            )

            if not cleaned_text:
                continue

            cleaned_pages.append({
                "pdf_page": pdf_page_number,
                "text": cleaned_text
            })


    print("Total PDF pages:", total_pdf_pages)
    print("Skipped PDF pages:", sorted(SKIP_PDF_PAGES))
    print("Cleaned page records:", len(cleaned_pages))


### Remove Unrelated Page Content
- Defines cleaning logic for pages containing non-knowledge or administrative content.


In [ ]:
if REBUILD_INDEX:
    # clean unrelated page
    SKIP_PDF_PAGES = {
        1,  # Cover page
        3,  # Table of contents
        4   # Table of contents
    }


    cleaned_pages = []


    with pdfplumber.open(PDF_PATH) as pdf:

        total_pdf_pages = len(pdf.pages)

        for pdf_page_number, page in enumerate(
            pdf.pages,
            start=1
        ):

            if pdf_page_number in SKIP_PDF_PAGES:
                continue

            raw_text = page.extract_text() or ""

            cleaned_text = clean_pdf_page_text(
                raw_text
            )

            if not cleaned_text:
                continue

            cleaned_pages.append({
                "pdf_page": pdf_page_number,
                "text": cleaned_text
            })


    print("Total PDF pages:", total_pdf_pages)
    print("Skipped PDF pages:", sorted(SKIP_PDF_PAGES))
    print("Cleaned page records:", len(cleaned_pages))


### Build Cleaned Page Set
- Applies the page filters and text cleaning rules to create the cleaned document pages.


In [ ]:
if REBUILD_INDEX:
    # cleaned page
    for record in cleaned_pages[:3]:

        print("=" * 80)
        print("PDF page:", record["pdf_page"])
        print("=" * 80)

        print(
            record["text"][:1500]
        )

        print()


## 2. Structure Extraction
- Detects handbook chapter and subsection boundaries.
- Converts cleaned page text into structured sections for chunking.


### Detect Section Headings
- Uses regular expressions to identify numbered handbook section headings.


In [ ]:
if REBUILD_INDEX:
    # use regex to identify structure
    HEADING_PATTERN = re.compile(
        r"^(\d+)\.(\d+)\s+(.+?)\s*$"
    )


    def identify_heading(line):

        match = HEADING_PATTERN.match(
            line.strip()
        )

        if not match:
            return None

        chapter_number = match.group(1)
        subsection_number = match.group(2)
        title = match.group(3).strip()

        return {
            "chapter_number": chapter_number,
            "subsection_number": subsection_number,
            "section_number": (
                f"{chapter_number}.{subsection_number}"
            ),
            "title": title,
            "is_chapter_heading": (
                subsection_number == "0"
            )
        }


### Validate Heading Detection
- Tests the heading pattern before full structure extraction.


In [ ]:
if REBUILD_INDEX:
    # test correct identify
    heading_tests = [
        "1.0 Selamat Datang ke Klinik Chong",
        "1.1 Pengenalan Klinik",
        "5.7 Pembatalan dan penjadualan semula",
        "COVID-19 dan gejala pernafasan",
        "50400 Kuala Lumpur"
    ]


    for test in heading_tests:

        print(
            test,
            "->",
            identify_heading(test)
        )


### Reconstruct Paragraph Text
- Combines line-level PDF text into coherent paragraph content.


In [ ]:
if REBUILD_INDEX:
    # combine the whole paragraph as paragraph but not line by line
    def rebuild_section_content(raw_lines):

        blocks = []
        current_block = []
        current_type = None

        def save_current_block():

            nonlocal current_block
            nonlocal current_type

            if not current_block:
                return

            text = " ".join(current_block)

            text = re.sub(
                r"\s+",
                " ",
                text
            ).strip()

            if text:
                blocks.append({
                    "type": current_type,
                    "text": text
                })

            current_block = []
            current_type = None

        for line in raw_lines:

            line = line.strip()

            if not line:
                continue

            # Start of a bullet point
            if line.startswith("•"):

                save_current_block()

                current_type = "bullet"
                current_block = [
                    line
                ]

                continue

            # Continue a wrapped bullet
            if current_type == "bullet":

                previous_text = " ".join(
                    current_block
                ).strip()

                # If previous bullet has not ended,
                # this line is treated as its continuation
                if not re.search(
                    r"[.!?;:]$",
                    previous_text
                ):

                    current_block.append(line)
                    continue

                # Previous bullet is complete
                save_current_block()

            # Normal paragraph
            if current_type != "paragraph":

                save_current_block()

                current_type = "paragraph"
                current_block = []

            current_block.append(line)

        save_current_block()

        # Keep bullets on separate lines
        final_parts = []

        for block in blocks:

            if block["type"] == "bullet":
                final_parts.append(
                    block["text"]
                )
            else:
                final_parts.append(
                    block["text"]
                )

        return "\n".join(final_parts)


### Extract Document Structure
- Parses cleaned pages into chapter, section, title and content fields.


In [ ]:
if REBUILD_INDEX:
    # extract document structure
    def extract_document_structure(cleaned_pages):

        structured_sections = []

        current_section = None

        current_chapter_number = None
        current_chapter_title = None

        def save_current_section():

            nonlocal current_section

            if current_section is None:
                return

            raw_lines = current_section.pop(
                "raw_lines"
            )

            content = rebuild_section_content(
                raw_lines
            )

            current_section["content"] = content

            # Do not save empty headings
            if content.strip():
                structured_sections.append(
                    current_section
                )

            current_section = None

        for page_record in cleaned_pages:

            pdf_page = page_record["pdf_page"]
            lines = page_record["text"].splitlines()

            for line in lines:

                line = line.strip()

                if not line:
                    continue

                heading = identify_heading(line)

                # --------------------------------
                # New numbered heading detected
                # --------------------------------
                if heading is not None:

                    save_current_section()

                    if heading["is_chapter_heading"]:

                        current_chapter_number = (
                            heading["chapter_number"]
                        )

                        current_chapter_title = (
                            heading["title"]
                        )

                    current_section = {
                        "chapter_number": (
                            current_chapter_number
                            or heading["chapter_number"]
                        ),
                        "chapter_title": (
                            current_chapter_title
                            or heading["title"]
                        ),
                        "section_number": (
                            heading["section_number"]
                        ),
                        "section_title": (
                            heading["title"]
                        ),
                        "page_start": pdf_page,
                        "page_end": pdf_page,
                        "raw_lines": []
                    }

                    continue

                # --------------------------------
                # Handle Kata Alu-aluan
                # --------------------------------
                if current_section is None:

                    current_section = {
                        "chapter_number": "0",
                        "chapter_title": "Front Matter",
                        "section_number": "0.1",
                        "section_title": "Kata Alu-aluan",
                        "page_start": pdf_page,
                        "page_end": pdf_page,
                        "raw_lines": []
                    }

                current_section["page_end"] = (
                    pdf_page
                )

                current_section["raw_lines"].append(
                    line
                )

        save_current_section()

        return structured_sections


### Build Structured Sections
- Creates the final structured section records used by the chunking stage.


In [ ]:
if REBUILD_INDEX:
    structured_sections = (
        extract_document_structure(
            cleaned_pages
        )
    )

    print(
        "Total structured sections:",
        len(structured_sections)
    )


### Inspect Extracted Sections
- Displays the extracted handbook structure in tabular form.


In [ ]:
if REBUILD_INDEX:
    sections_df = pd.DataFrame(
        structured_sections
    )

    sections_df["character_count"] = (
        sections_df["content"].str.len()
    )

    sections_df["word_count"] = (
        sections_df["content"].str.split().str.len()
    )

    display(
        sections_df[
            [
                "chapter_number",
                "chapter_title",
                "section_number",
                "section_title",
                "page_start",
                "page_end",
                "character_count",
                "word_count"
            ]
        ]
    )


### Review Extracted Content
- Prints section content for manual inspection.


In [ ]:
if REBUILD_INDEX:
    for section in structured_sections:

        print(
            f"{section['section_number']:<5} | "
            f"{section['section_title']:<55} | "
            f"PDF pages "
            f"{section['page_start']}"
            f"-{section['page_end']}"
        )


### Validate Extracted Structure
- Checks section numbering and content completeness before chunking.


In [ ]:
if REBUILD_INDEX:
    # structure validation
    duplicate_sections = (
        sections_df[
            sections_df["section_number"]
            .duplicated(keep=False)
        ]
    )


    empty_sections = (
        sections_df[
            sections_df["content"]
            .str.strip()
            .eq("")
        ]
    )


    invalid_page_ranges = (
        sections_df[
            sections_df["page_start"]
            >
            sections_df["page_end"]
        ]
    )


    print(
        "Duplicate section numbers:",
        len(duplicate_sections)
    )

    print(
        "Empty sections:",
        len(empty_sections)
    )

    print(
        "Invalid page ranges:",
        len(invalid_page_ranges)
    )


## 3. Structure-Aware Chunking
- Uses handbook subsections as semantic chunks instead of arbitrary fixed-length windows.
- Assigns deterministic chunk IDs for stable indexing.


### Measure Section Length
- Measures section sizes before chunk creation.


In [ ]:
if REBUILD_INDEX:
    # word count each section
    section_length_df = (
        sections_df[
            [
                "section_number",
                "section_title",
                "word_count",
                "character_count"
            ]
        ]
        .sort_values(
            "word_count",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(
        section_length_df.head(15)
    )

    print(
        "Minimum words:",
        sections_df["word_count"].min()
    )

    print(
        "Median words:",
        int(
            sections_df["word_count"].median()
        )
    )

    print(
        "Maximum words:",
        sections_df["word_count"].max()
    )

    print(
        "Sections above 250 words:",
        (
            sections_df["word_count"] > 250
        ).sum()
    )


### Define Stable Chunk IDs
- Creates deterministic IDs from handbook section numbers.


In [ ]:
if REBUILD_INDEX:
    # build chunk id
    def create_chunk_id(section_number):

        safe_section_number = (
            section_number.replace(".", "_")
        )

        return (
            "klinik_chong_handbook_ms_"
            f"sec_{safe_section_number}"
        )


### Test Chunk ID Generation
- Verifies the deterministic chunk ID format.


In [ ]:
if REBUILD_INDEX:
    print(create_chunk_id("1.1"))
    print(create_chunk_id("5.7"))


### Define Structure-Aware Chunking
- Converts structured handbook sections into retrieval-ready chunks.


In [ ]:
if REBUILD_INDEX:
    # structure chunking function
    def create_structure_chunks(
        structured_sections
    ):

        chunks = []

        for section in structured_sections:

            chunk_id = create_chunk_id(
                section["section_number"]
            )

            chapter_heading = (
                f"Chapter: "
                f"{section['chapter_number']} - "
                f"{section['chapter_title']}"
            )

            section_heading = (
                f"Section: "
                f"{section['section_number']} - "
                f"{section['section_title']}"
            )

            embedding_text = (
                f"{chapter_heading}\n"
                f"{section_heading}\n\n"
                f"{section['content']}"
            )

            chunk = {
                "chunk_id": chunk_id,

                "chapter_number": (
                    section["chapter_number"]
                ),

                "chapter_title": (
                    section["chapter_title"]
                ),

                "section_number": (
                    section["section_number"]
                ),

                "section_title": (
                    section["section_title"]
                ),

                "page_start": (
                    section["page_start"]
                ),

                "page_end": (
                    section["page_end"]
                ),

                # Cleaned original section content
                "content": (
                    section["content"]
                ),

                # Text that will be embedded later
                "embedding_text": (
                    embedding_text
                )
            }

            chunks.append(chunk)

        return chunks


### Create Structure-Aware Chunks
- Generates the complete chunk collection.


In [ ]:
if REBUILD_INDEX:
    structure_chunks = (
        create_structure_chunks(
            structured_sections
        )
    )

    print(
        "Total structure chunks:",
        len(structure_chunks)
    )


### Inspect Chunk Table
- Summarizes chunk IDs, sections, titles and content lengths.


In [ ]:
if REBUILD_INDEX:
    chunks_df = pd.DataFrame(
        structure_chunks
    )

    chunks_df["content_word_count"] = (
        chunks_df["content"]
        .str.split()
        .str.len()
    )

    chunks_df["embedding_word_count"] = (
        chunks_df["embedding_text"]
        .str.split()
        .str.len()
    )

    chunks_df["content_character_count"] = (
        chunks_df["content"]
        .str.len()
    )

    display(
        chunks_df[
            [
                "chunk_id",
                "section_number",
                "section_title",
                "page_start",
                "page_end",
                "content_word_count",
                "embedding_word_count"
            ]
        ]
    )


### Inspect Chunk IDs
- Reviews generated chunk identifiers and section mapping.


In [ ]:
if REBUILD_INDEX:
    # check each chunk id
    chunk_index = 25

    chunk = structure_chunks[
        chunk_index
    ]

    print("=" * 80)
    print("Chunk index:", chunk_index)
    print("Chunk ID:", chunk["chunk_id"])
    print("Chapter:", chunk["chapter_title"])

    print(
        "Section:",
        chunk["section_number"],
        chunk["section_title"]
    )

    print(
        "PDF pages:",
        chunk["page_start"],
        "-",
        chunk["page_end"]
    )

    print("=" * 80)
    print("\nEMBEDDING TEXT:\n")
    print(chunk["embedding_text"])


### Inspect Largest Chunk
- Identifies the largest chunk for token-length and retrieval checks.


In [ ]:
if REBUILD_INDEX:
    # largest chunk
    largest_chunk = max(
        structure_chunks,
        key=lambda chunk: len(
            chunk["content"].split()
        )
    )

    print(
        "Largest chunk:",
        largest_chunk["section_number"],
        largest_chunk["section_title"]
    )

    print(
        "Word count:",
        len(
            largest_chunk["content"].split()
        )
    )

    print(
        "PDF pages:",
        largest_chunk["page_start"],
        "-",
        largest_chunk["page_end"]
    )

    print("\nContent:\n")
    print(largest_chunk["embedding_text"])


### Validate Chunk Set
- Checks chunk uniqueness and required fields before metadata assignment.


In [ ]:
if REBUILD_INDEX:
    # chunk validation
    chunk_ids = [
        chunk["chunk_id"]
        for chunk in structure_chunks
    ]


    empty_chunks = [
        chunk["chunk_id"]
        for chunk in structure_chunks
        if not chunk["content"].strip()
    ]


    duplicate_chunk_ids = (
        len(chunk_ids)
        -
        len(set(chunk_ids))
    )


    missing_section_titles = [
        chunk["chunk_id"]
        for chunk in structure_chunks
        if not chunk["section_title"].strip()
    ]


    print(
        "Total chunks:",
        len(structure_chunks)
    )

    print(
        "Empty chunks:",
        len(empty_chunks)
    )

    print(
        "Duplicate chunk IDs:",
        duplicate_chunk_ids
    )

    print(
        "Missing section titles:",
        len(missing_section_titles)
    )


## 4. Metadata Assignment
- Adds retrieval metadata such as chapter, subsection, knowledge domain, content type and emergency relevance.


### Define Knowledge Domain Mapping
- Maps handbook chapters to high-level clinic knowledge domains.


In [ ]:
if REBUILD_INDEX:
    # Cchapter domain mapping
    CHAPTER_DOMAIN_MAP = {
        "0": "front_matter",
        "1": "general_information",
        "2": "clinic_information",
        "3": "patient_rights_and_responsibilities",
        "4": "clinic_visit",
        "5": "appointment",
        "6": "chatbot_information",
        "7": "general_health_information"
    }


    def assign_knowledge_domain(
        chapter_number
    ):

        return CHAPTER_DOMAIN_MAP.get(
            str(chapter_number),
            "unknown"
        )


### Define Content Types
- Assigns content-type labels to chunks based on their section purpose.


In [ ]:
if REBUILD_INDEX:
    # assign content type
    def assign_content_type(
        section_number,
        chapter_number
    ):

        # Disclaimer and chatbot limitations
        if section_number in {
            "1.4",
            "6.4",
            "6.6"
        }:
            return "limitation_and_safety"

        # Clinic contact/location/hours
        if section_number in {
            "2.3",
            "2.4",
            "2.5"
        }:
            return "clinic_operational_information"

        if chapter_number == "3":
            return "patient_policy"

        if chapter_number == "4":
            return "visit_procedure"

        if chapter_number == "5":
            return "appointment_policy_and_procedure"

        if chapter_number == "6":
            return "chatbot_information"

        if chapter_number == "7":
            return "general_health_guidance"

        if chapter_number == "0":
            return "front_matter"

        return "general_information"


### Mark Emergency-Related Sections
- Flags sections containing emergency or urgent-care information.


In [ ]:
if REBUILD_INDEX:
    # emergency relatyed section
    EMERGENCY_RELATED_SECTIONS = {
        "1.4",
        "2.2",
        "6.4",
        "6.6",
        "7.5",
        "7.6",
        "7.7"
    }


    def is_emergency_related(
        section_number
    ):

        return (
            section_number
            in EMERGENCY_RELATED_SECTIONS
        )


### Attach Metadata to Chunks
- Adds all metadata fields to each structure-aware chunk.


In [ ]:
if REBUILD_INDEX:
    # adding metadata into each chunk
    DOCUMENT_ID = (
        "klinik_chong_patient_handbook_ms"
    )

    SOURCE_FILE = (
        "Klinik Chong Buku Panduan Pesakit.pdf"
    )

    DOCUMENT_VERSION = "1.0"

    DOCUMENT_LANGUAGE = "ms"

    DOCUMENT_TYPE = "patient_handbook"


    def assign_metadata_to_chunks(
        structure_chunks
    ):

        chunks_with_metadata = []

        for chunk in structure_chunks:

            chapter_number = str(
                chunk["chapter_number"]
            )

            section_number = str(
                chunk["section_number"]
            )

            metadata = {
                "document_id": DOCUMENT_ID,

                "source_file": SOURCE_FILE,

                "document_type": DOCUMENT_TYPE,

                "document_version": (
                    DOCUMENT_VERSION
                ),

                "language": DOCUMENT_LANGUAGE,

                "chapter_number": (
                    chapter_number
                ),

                "chapter_title": (
                    chunk["chapter_title"]
                ),

                "section_number": (
                    section_number
                ),

                "section_title": (
                    chunk["section_title"]
                ),

                "knowledge_domain": (
                    assign_knowledge_domain(
                        chapter_number
                    )
                ),

                "content_type": (
                    assign_content_type(
                        section_number,
                        chapter_number
                    )
                ),

                "page_start": int(
                    chunk["page_start"]
                ),

                "page_end": int(
                    chunk["page_end"]
                ),

                # Handbook is static knowledge
                "is_dynamic": False,

                "is_emergency_related": (
                    is_emergency_related(
                        section_number
                    )
                ),

                # Kata Alu-aluan is retained,
                # but excluded from vector index
                "include_in_index": (
                    section_number != "0.1"
                )
            }

            enriched_chunk = {
                **chunk,
                "metadata": metadata
            }

            chunks_with_metadata.append(
                enriched_chunk
            )

        return chunks_with_metadata


### Inspect Metadata Assignment
- Checks that metadata was attached correctly.


In [ ]:
if REBUILD_INDEX:
    # check whether adding in?
    chunks_with_metadata = (
        assign_metadata_to_chunks(
            structure_chunks
        )
    )

    print(
        "Chunks with metadata:",
        len(chunks_with_metadata)
    )


### Validate One Metadata Record
- Validates required metadata fields on a sample chunk.


In [ ]:
if REBUILD_INDEX:
    # validate metadata for the chunk
    section_to_check = "5.7"

    selected_chunk = next(
        chunk
        for chunk in chunks_with_metadata
        if chunk["section_number"] == section_to_check
    )

    print("Chunk ID:")
    print(selected_chunk["chunk_id"])

    print("\nMetadata:")

    for key, value in selected_chunk["metadata"].items():
        print(f"{key:<25}: {value}")


### Create Metadata Table
- Converts chunk metadata into a DataFrame for inspection.


In [ ]:
if REBUILD_INDEX:
    metadata_rows = []

    for chunk in chunks_with_metadata:

        row = {
            "chunk_id": chunk["chunk_id"],
            **chunk["metadata"]
        }

        metadata_rows.append(row)


    metadata_df = pd.DataFrame(
        metadata_rows
    )


    display(
        metadata_df[
            [
                "chunk_id",
                "section_number",
                "section_title",
                "knowledge_domain",
                "content_type",
                "page_start",
                "page_end",
                "is_emergency_related",
                "include_in_index"
            ]
        ]
    )


### Summarize Knowledge Domains
- Counts chunks by assigned knowledge domain.


In [ ]:
if REBUILD_INDEX:
    domain_counts = (
        metadata_df[
            "knowledge_domain"
        ]
        .value_counts()
        .rename_axis(
            "knowledge_domain"
        )
        .reset_index(
            name="chunk_count"
        )
    )

    display(domain_counts)


### Validate Chroma Metadata Compatibility
- Ensures metadata values use data types supported by ChromaDB.


In [ ]:
if REBUILD_INDEX:
    ALLOWED_METADATA_TYPES = (
        str,
        int,
        float,
        bool
    )


    invalid_metadata_values = []


    for chunk in chunks_with_metadata:

        for key, value in (
            chunk["metadata"].items()
        ):

            if not isinstance(
                value,
                ALLOWED_METADATA_TYPES
            ):

                invalid_metadata_values.append({
                    "chunk_id": (
                        chunk["chunk_id"]
                    ),
                    "metadata_key": key,
                    "value": value,
                    "type": type(value).__name__
                })


    missing_metadata = []


    required_metadata_fields = {
        "document_id",
        "source_file",
        "document_type",
        "document_version",
        "language",
        "chapter_number",
        "chapter_title",
        "section_number",
        "section_title",
        "knowledge_domain",
        "content_type",
        "page_start",
        "page_end",
        "is_dynamic",
        "is_emergency_related",
        "include_in_index"
    }


    for chunk in chunks_with_metadata:

        available_fields = set(
            chunk["metadata"].keys()
        )

        missing_fields = (
            required_metadata_fields
            - available_fields
        )

        if missing_fields:

            missing_metadata.append({
                "chunk_id": chunk["chunk_id"],
                "missing_fields": (
                    sorted(missing_fields)
                )
            })


    print(
        "Invalid metadata values:",
        len(invalid_metadata_values)
    )

    print(
        "Chunks with missing metadata:",
        len(missing_metadata)
    )

    print(
        "Total chunks:",
        len(chunks_with_metadata)
    )

    print(
        "Chunks included in index:",
        sum(
            chunk["metadata"][
                "include_in_index"
            ]
            for chunk
            in chunks_with_metadata
        )
    )


## 5. Embedding
- Encodes handbook chunks using the multilingual `BAAI/bge-m3` embedding model.
- Validates token length, embedding dimensions and vector normalization.


### Select Compute Device
- Uses GPU when available, otherwise falls back to CPU.


In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if device == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

### Load BGE-M3 Embedding Model
- Loads the multilingual embedding model used for both documents and queries.


In [ ]:
# load model BGE-M3
EMBEDDING_MODEL_NAME = (
    "BAAI/bge-m3"
)


embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device
)


print(
    "Embedding model:",
    EMBEDDING_MODEL_NAME
)

print(
    "Maximum sequence length:",
    embedding_model.max_seq_length
)

print(
    "Embedding dimension:",
    (
        embedding_model
        .get_sentence_embedding_dimension()
    )
)

### Select Chunks for Indexing
- Filters the final chunk set that will be embedded and stored.


In [ ]:
if REBUILD_INDEX:
    # choose the chunk put in the embedding model
    indexed_chunks = [
        chunk
        for chunk in chunks_with_metadata
        if chunk["metadata"][
            "include_in_index"
        ]
    ]


    print(
        "All chunks:",
        len(chunks_with_metadata)
    )

    print(
        "Chunks selected for embedding:",
        len(indexed_chunks)
    )


    excluded_chunks = [
        chunk
        for chunk in chunks_with_metadata
        if not chunk["metadata"][
            "include_in_index"
        ]
    ]


    print("\nExcluded chunks:")

    for chunk in excluded_chunks:

        print(
            chunk["section_number"],
            "-",
            chunk["section_title"]
        )


### Check Embedding Token Lengths
- Verifies that chunk token lengths remain within the embedding model limit.


In [ ]:
if REBUILD_INDEX:
    # token length validation
    token_length_records = []


    for chunk in indexed_chunks:

        encoded = (
            embedding_model.tokenizer(
                chunk["embedding_text"],
                add_special_tokens=True,
                truncation=False
            )
        )

        token_count = len(
            encoded["input_ids"]
        )

        token_length_records.append({
            "chunk_id": (
                chunk["chunk_id"]
            ),

            "section_number": (
                chunk["section_number"]
            ),

            "section_title": (
                chunk["section_title"]
            ),

            "token_count": token_count,

            "model_limit": (
                embedding_model
                .max_seq_length
            ),

            "within_limit": (
                token_count
                <=
                embedding_model
                .max_seq_length
            )
        })


    token_length_df = pd.DataFrame(
        token_length_records
    )


### Inspect Longest Tokenized Chunks
- Reviews chunks with the highest token counts.


In [ ]:
if REBUILD_INDEX:
    display(
        token_length_df
        .sort_values(
            "token_count",
            ascending=False
        )
        .head(15)
    )


### Summarize Token-Length Validation
- Reports whether any chunk exceeds the allowed token length.


In [ ]:
if REBUILD_INDEX:
    over_limit_chunks = (
        token_length_df[
            token_length_df[
                "within_limit"
            ]
            == False
        ]
    )


    print(
        "Total chunks checked:",
        len(token_length_df)
    )

    print(
        "Minimum token count:",
        token_length_df[
            "token_count"
        ].min()
    )

    print(
        "Average token count:",
        round(
            token_length_df[
                "token_count"
            ].mean(),
            2
        )
    )

    print(
        "Maximum token count:",
        token_length_df[
            "token_count"
        ].max()
    )

    print(
        "Chunks exceeding model limit:",
        len(over_limit_chunks)
    )


### Prepare Embedding Inputs
- Builds the text inputs passed to the embedding model.


In [ ]:
if REBUILD_INDEX:
    # prepare embedding text
    embedding_texts = [
        chunk["embedding_text"]
        for chunk in indexed_chunks
    ]


    print(
        "Number of embedding texts:",
        len(embedding_texts)
    )

    print("\nFirst embedding text:\n")

    print(
        embedding_texts[0][
            :1500
        ]
    )


### Generate Document Embeddings
- Encodes all selected chunks into normalized dense vectors.


In [ ]:
if REBUILD_INDEX:
    # generate embedding
    embeddings = (
        embedding_model.encode(
            embedding_texts,

            batch_size=8,

            show_progress_bar=True,

            convert_to_numpy=True,

            normalize_embeddings=True
        )
    )


### Inspect Embedding Output
- Checks the number and shape of generated embeddings.


In [ ]:
if REBUILD_INDEX:
    print(
        "Embedding array type:",
        type(embeddings)
    )

    print(
        "Embedding shape:",
        embeddings.shape
    )

    print(
        "Embedding dtype:",
        embeddings.dtype
    )


### Inspect One Embedding
- Displays a sample vector for sanity checking.


In [ ]:
if REBUILD_INDEX:
    # check embedding
    chunk_index = 0


    print(
        "Chunk ID:",
        indexed_chunks[
            chunk_index
        ]["chunk_id"]
    )

    print(
        "Section:",
        indexed_chunks[
            chunk_index
        ]["section_number"],
        "-",
        indexed_chunks[
            chunk_index
        ]["section_title"]
    )

    print(
        "\nFirst 10 embedding values:"
    )

    print(
        embeddings[
            chunk_index
        ][:10]
    )


### Validate Embedding Dimensions
- Confirms that every indexed chunk has the expected BGE-M3 vector dimension.


In [ ]:
if REBUILD_INDEX:
    expected_chunk_count = len(
        indexed_chunks
    )

    expected_dimension = (
        embedding_model
        .get_sentence_embedding_dimension()
    )


    has_nan = np.isnan(
        embeddings
    ).any()


    has_infinity = np.isinf(
        embeddings
    ).any()


    correct_chunk_count = (
        embeddings.shape[0]
        ==
        expected_chunk_count
    )


    correct_dimension = (
        embeddings.shape[1]
        ==
        expected_dimension
    )


    print(
        "Expected chunks:",
        expected_chunk_count
    )

    print(
        "Generated embeddings:",
        embeddings.shape[0]
    )

    print(
        "Expected dimension:",
        expected_dimension
    )

    print(
        "Actual dimension:",
        embeddings.shape[1]
    )

    print(
        "Contains NaN:",
        has_nan
    )

    print(
        "Contains infinity:",
        has_infinity
    )

    print(
        "Correct chunk count:",
        correct_chunk_count
    )

    print(
        "Correct embedding dimension:",
        correct_dimension
    )


### Validate Vector Normalization
- Calculates vector norms to verify normalized embeddings.


In [ ]:
if REBUILD_INDEX:
    embedding_norms = np.linalg.norm(
        embeddings,
        axis=1
    )


    print(
        "Minimum vector norm:",
        embedding_norms.min()
    )

    print(
        "Maximum vector norm:",
        embedding_norms.max()
    )

    print(
        "Average vector norm:",
        embedding_norms.mean()
    )


    print(
        "All vectors approximately normalized:",
        np.allclose(
            embedding_norms,
            1.0,
            atol=1e-5
        )
    )


### Inspect Vector Norms
- Reviews normalized vector lengths across the embedding set.


In [ ]:
if REBUILD_INDEX:
    # check normalized vector length
    # allowing cosine similarity to compare their semantic meanings fairly
    embedding_norms = np.linalg.norm(
       embeddings,
       axis=1
    )


    print(
       "Minimum vector norm:",
       embedding_norms.min()
    )

    print(
       "Maximum vector norm:",
       embedding_norms.max()
    )

    print(
       "Average vector norm:",
       embedding_norms.mean()
    )


    print(
       "All vectors approximately normalized:",
       np.allclose(
           embedding_norms,
           1.0,
           atol=1e-5
       )
    )


## 6. Store in ChromaDB
- Persists handbook vectors and metadata in the project ChromaDB collection.
- Uses stable chunk IDs so records can be safely updated without duplicating entries.


### Check ChromaDB Version
- Records the installed ChromaDB version for reproducibility.


In [ ]:
print(
    "ChromaDB version:",
    chromadb.__version__
)

### Confirm Google Drive Mount
- Confirms that the persistent database and ChromaDB paths are available.


In [ ]:
print("Google Drive available:", FYP2_ROOT.exists())


### Resolve Persistent ChromaDB Path
- Defines the persistent storage location for the Klinik Chong vector database.


In [ ]:
# Reuse paths defined in the Run-All configuration cell.
if not CHROMA_DB_PATH.exists():
    raise FileNotFoundError(f"ChromaDB folder not found: {CHROMA_DB_PATH}")

print("ChromaDB path:", CHROMA_DB_PATH)
print("Path exists:", CHROMA_DB_PATH.exists())


### Open ChromaDB Collection
- Creates or opens the persistent handbook collection.


In [ ]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
COLLECTION_NAME = "klinik_chong_handbook"

if REBUILD_INDEX:
    collection = chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={
            "description": "Klinik Chong patient handbook Malay knowledge base",
            "embedding_model": "BAAI/bge-m3",
            "embedding_dimension": 1024,
            "hnsw:space": "cosine",
        },
        embedding_function=None,
    )
else:
    try:
        collection = chroma_client.get_collection(
            name=COLLECTION_NAME,
            embedding_function=None,
        )
    except Exception as error:
        raise RuntimeError(
            f"Existing ChromaDB collection '{COLLECTION_NAME}' was not found. "
            "Set REBUILD_INDEX=True only when rebuilding from the handbook."
        ) from error

if collection.count() == 0:
    raise RuntimeError("The Klinik Chong ChromaDB collection is empty.")

print("Collection name:", collection.name)
print("Existing records:", collection.count())
print("Ingestion skipped:", not REBUILD_INDEX)


### Prepare ChromaDB Records
- Prepares document text, IDs, embeddings and metadata for storage.


In [ ]:
if REBUILD_INDEX:
    # prepare chroma data
    chroma_ids = [
        chunk["chunk_id"]
        for chunk in indexed_chunks
    ]


    chroma_documents = [
        chunk["content"]
        for chunk in indexed_chunks
    ]


    chroma_metadatas = [
        chunk["metadata"]
        for chunk in indexed_chunks
    ]


    chroma_embeddings = (
        embeddings.tolist()
    )


    print(
        "IDs:",
        len(chroma_ids)
    )

    print(
        "Documents:",
        len(chroma_documents)
    )

    print(
        "Metadatas:",
        len(chroma_metadatas)
    )

    print(
        "Embeddings:",
        len(chroma_embeddings)
    )

    print(
        "Embedding dimension:",
        len(chroma_embeddings[0])
    )


### Store or Update Vector Records
- Upserts the prepared handbook records into ChromaDB.


In [ ]:
if REBUILD_INDEX:
    # store in chromaDB
    collection.upsert(
        ids=chroma_ids,
        documents=chroma_documents,
        metadatas=chroma_metadatas,
        embeddings=chroma_embeddings
    )


    print(
        "Stored records:",
        collection.count()
    )


### Read Back One Stored Record
- Reads a sample record to verify persistence.


In [ ]:
# Read back one existing record without depending on the ingestion variables.
stored_record = collection.get(
    limit=1,
    include=["documents", "metadatas", "embeddings"],
)
if not stored_record["ids"]:
    raise RuntimeError("No ChromaDB record is available for validation.")

print("Sample record ID:", stored_record["ids"][0])
print("Section:", stored_record["metadatas"][0].get("section_number"))
print("Title:", stored_record["metadatas"][0].get("section_title"))
print("Stored embedding dimension:", len(stored_record["embeddings"][0]))


### Validate Stored Record Set
- Confirms that the expected chunk records are present in the collection.


In [ ]:
if REBUILD_INDEX:
    # store successsfully
    stored_data = collection.get()


    stored_ids = set(
        stored_data["ids"]
    )

    expected_ids = set(
        chroma_ids
    )


    missing_ids = (
        expected_ids
        -
        stored_ids
    )


    unexpected_ids = (
        stored_ids
        -
        expected_ids
    )


    print(
        "Expected IDs:",
        len(expected_ids)
    )

    print(
        "Stored IDs:",
        len(stored_ids)
    )

    print(
        "Missing IDs:",
        len(missing_ids)
    )

    print(
        "Unexpected IDs:",
        len(unexpected_ids)
    )


    if missing_ids:

        print(
            "\nMissing:",
            sorted(missing_ids)
        )


    if unexpected_ids:

        print(
            "\nUnexpected:",
            sorted(unexpected_ids)
        )


## 7. Index Validation
- Validates that indexed chunks can retrieve themselves and defines the reusable retrieval functions.


### Run Self-Retrieval Validation
- Checks whether each indexed chunk is returned as the top result for its own content.


In [ ]:
if REBUILD_INDEX:
    self_retrieval_results = (
        collection.query(
            query_embeddings=(
                embeddings.tolist()
            ),

            n_results=1,

            include=[
                "metadatas",
                "distances"
            ]
        )
    )


    self_retrieval_errors = []


    for index, expected_chunk in enumerate(
        indexed_chunks
    ):

        expected_id = (
            expected_chunk["chunk_id"]
        )

        returned_id = (
            self_retrieval_results[
                "ids"
            ][index][0]
        )

        distance = (
            self_retrieval_results[
                "distances"
            ][index][0]
        )

        if returned_id != expected_id:

            self_retrieval_errors.append({
                "expected_id": expected_id,
                "returned_id": returned_id,
                "distance": distance
            })


    print(
        "Chunks checked:",
        len(indexed_chunks)
    )

    print(
        "Correct self-retrieval:",
        (
            len(indexed_chunks)
            -
            len(self_retrieval_errors)
        )
    )

    print(
        "Self-retrieval errors:",
        len(self_retrieval_errors)
    )


### Define Retrieval Function
- Queries ChromaDB and returns structured retrieval results with similarity metadata.


In [ ]:
# build retrival function
def retrieve_chunks(
    query,
    top_k=3,
    where=None
):

    query_embedding = (
        create_query_embedding(
            query
        )
    )

    query_arguments = {
        "query_embeddings": (
            query_embedding.tolist()
        ),

        "n_results": top_k,

        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    if where is not None:

        query_arguments["where"] = where

    results = collection.query(
        **query_arguments
    )

    retrieved_chunks = []

    for rank in range(
        len(results["ids"][0])
    ):

        distance = float(
            results[
                "distances"
            ][0][rank]
        )

        metadata = (
            results[
                "metadatas"
            ][0][rank]
        )

        retrieved_chunks.append({
            "rank": rank + 1,

            "chunk_id": (
                results[
                    "ids"
                ][0][rank]
            ),

            "section_number": (
                metadata[
                    "section_number"
                ]
            ),

            "section_title": (
                metadata[
                    "section_title"
                ]
            ),

            "knowledge_domain": (
                metadata[
                    "knowledge_domain"
                ]
            ),

            "distance": distance,

            "similarity": (
                1.0 - distance
            ),

            "document": (
                results[
                    "documents"
                ][0][rank]
            ),

            "metadata": metadata
        })

    return retrieved_chunks

### Define Query Embedding Function
- Encodes user queries with the same BGE-M3 model used for document vectors.


In [ ]:
def create_query_embedding(
    query
):

    query_embedding = (
        embedding_model.encode(
            [query],

            convert_to_numpy=True,

            normalize_embeddings=True
        )
    )

    return query_embedding

## 8. Baseline Retrieval Evaluation
- Performs an initial multilingual retrieval check before the full RAG pipeline.
- Measures Hit@1, Hit@3 and Mean Reciprocal Rank (MRR) on the prepared test cases.


### Test Malay Retrieval
- Runs a manual Malay retrieval sanity check.


In [ ]:
if RUN_LOCAL_RETRIEVAL_EVALUATION:
    query = "klinik chong ada chatbot?"
    retrieved_chunks = retrieve_chunks(query=query, top_k=3)
    print("Query:", query)
    for result in retrieved_chunks:
        print(
            f"Rank {result['rank']} | Section {result['section_number']} | "
            f"{result['section_title']} | Similarity: {result['similarity']:.4f}"
        )
else:
    retrieved_chunks = []
    print("SKIPPED: manual Malay retrieval example (no embedding call).")


### Inspect Retrieved Documents
- Displays the retrieved sections for manual verification.


In [ ]:
if RUN_LOCAL_RETRIEVAL_EVALUATION:
    for result in retrieved_chunks:
        print("=" * 80)
        print("Rank:", result["rank"])
        print("Section:", result["section_number"], "-", result["section_title"])
        print("Similarity:", round(result["similarity"], 4))
        print("\nDocument:\n")
        print(result["document"][:1000])
else:
    print("SKIPPED: retrieved-chunk preview.")


### Test Chinese Retrieval
- Runs a manual Chinese retrieval sanity check.


In [ ]:
if RUN_LOCAL_RETRIEVAL_EVALUATION:
    query = "我发烧怎么办？"
    results = retrieve_chunks(query=query, top_k=3)
    print("Query:", query)
    for result in results:
        print(
            f"Rank {result['rank']} | {result['section_number']} | "
            f"{result['section_title']} | Similarity: {result['similarity']:.4f}"
        )
else:
    results = []
    print("SKIPPED: manual Chinese retrieval example (no embedding call).")


### ChromaDB Retrieval Evaluation - 200 Knowledge-Base Questions
- Contains 200 questions covering all 40 handbook subsections.
- Includes 34 user-provided casual Malay, Chinese and mixed-language questions.
- Uses the expected subsection to calculate Hit@1, Hit@3 and MRR.


In [ ]:
retrieval_test_cases = [{'test_id': 'KC-RAG-001',
  'language': 'chinese',
  'expected_sections': ['1.1'],
  'query': '庄诊所是一家提供哪一类医疗服务的诊所？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-002',
  'language': 'malay',
  'expected_sections': ['1.1'],
  'query': 'Apakah fokus utama perkhidmatan yang disediakan oleh Klinik Chong?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-003',
  'language': 'chinese',
  'expected_sections': ['1.1'],
  'query': '庄诊所主要为当地社区提供哪些基本医疗服务？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-004',
  'language': 'malay',
  'expected_sections': ['1.1'],
  'query': 'Bagaimanakah chatbot membantu pesakit dalam urusan Klinik Chong?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-005',
  'language': 'chinese',
  'expected_sections': ['1.1'],
  'query': '诊所的预约机器人可以协助病人完成哪些事情？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-006',
  'language': 'chinese',
  'expected_sections': ['1.2'],
  'query': '这本庄诊所病人手册是为谁准备的？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-007',
  'language': 'malay',
  'expected_sections': ['1.2'],
  'query': 'Apakah tujuan utama Buku Panduan Pesakit Klinik Chong?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-008',
  'language': 'chinese',
  'expected_sections': ['1.2'],
  'query': '病人手册是否说明了看诊前、看诊期间和看诊后的事项？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-009',
  'language': 'malay',
  'expected_sections': ['1.2'],
  'query': 'Adakah buku panduan ini menerangkan hak dan tanggungjawab pesakit?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-010',
  'language': 'chinese',
  'expected_sections': ['1.2'],
  'query': '如果家属想了解诊所服务和预约流程，可以参考什么资料？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-011',
  'language': 'chinese',
  'expected_sections': ['1.3'],
  'query': '庄诊所支持病人使用哪些语言沟通？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-012',
  'language': 'malay',
  'expected_sections': ['1.3'],
  'query': 'Bolehkah chatbot memahami ayat yang mencampurkan Bahasa Melayu dan Bahasa Cina?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-013',
  'language': 'chinese',
  'expected_sections': ['1.3'],
  'query': '如果机器人无法确定病人使用的语言，它会怎么做？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-014',
  'language': 'malay',
  'expected_sections': ['1.3'],
  'query': 'Adakah pesakit boleh menggunakan bahasa yang paling selesa ketika bercakap dengan chatbot?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-015',
  'language': 'chinese',
  'expected_sections': ['1.3'],
  'query': '为了减少语言误解，病人在输入信息时应该注意什么？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-016',
  'language': 'chinese',
  'expected_sections': ['1.4'],
  'query': '聊天机器人提供的健康资料可以当作正式诊断吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-017',
  'language': 'malay',
  'expected_sections': ['1.4'],
  'query': 'Adakah chatbot boleh menggantikan pemeriksaan oleh pegawai perubatan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-018',
  'language': 'chinese',
  'expected_sections': ['1.4'],
  'query': '提交预约申请后，是否代表预约已经成功？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-019',
  'language': 'malay',
  'expected_sections': ['1.4'],
  'query': 'Apakah yang perlu dilakukan jika seseorang memerlukan bantuan perubatan segera?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-020',
  'language': 'chinese',
  'expected_sections': ['1.4'],
  'query': '机器人生成的临床摘要在使用前需要由谁确认？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-021',
  'language': 'chinese',
  'expected_sections': ['2.1'],
  'query': '庄诊所的宗旨是什么？',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-022',
  'language': 'malay',
  'expected_sections': ['2.1'],
  'query': 'apakah visi dan misi klinik chong?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-023',
  'language': 'chinese',
  'expected_sections': ['2.1'],
  'query': '诊所的使命是否包括保护病人的隐私和资料机密？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-024',
  'language': 'malay',
  'expected_sections': ['2.1'],
  'query': 'Bagaimanakah teknologi digunakan untuk menyokong misi Klinik Chong?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-025',
  'language': 'chinese',
  'expected_sections': ['2.1'],
  'query': '庄诊所在服务态度和环境方面有什么承诺？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-026',
  'language': 'chinese',
  'expected_sections': ['2.2'],
  'query': '庄诊所提供哪些基层医疗和普通医疗服务？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-027',
  'language': 'malay',
  'expected_sections': ['2.2'],
  'query': 'Adakah Klinik Chong menyediakan pemeriksaan tekanan darah dan suhu badan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-028',
  'language': 'chinese',
  'expected_sections': ['2.2'],
  'query': '诊所会在什么情况下提供转介信？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-029',
  'language': 'malay',
  'expected_sections': ['2.2'],
  'query': 'Bolehkah pesakit mendapatkan rawatan susulan di Klinik Chong?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-030',
  'language': 'chinese',
  'expected_sections': ['2.2'],
  'query': '庄诊所是否提供医院级别的紧急医疗服务？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-031',
  'language': 'chinese',
  'expected_sections': ['2.3'],
  'query': 'Klinik 在哪里?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-032',
  'language': 'malay',
  'expected_sections': ['2.3'],
  'query': 'Mana Klinik Chong',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-033',
  'language': 'chinese',
  'expected_sections': ['2.3'],
  'query': '病人可以使用哪些交通方式前往诊所？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-034',
  'language': 'malay',
  'expected_sections': ['2.3'],
  'query': 'Berapa awal pesakit digalakkan tiba sebelum waktu temu janji?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-035',
  'language': 'chinese',
  'expected_sections': ['2.3'],
  'query': '我可以通过聊天机器人取得诊所的路线和电子地图位置吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-036',
  'language': 'malay',
  'expected_sections': ['2.4'],
  'query': 'Macam mana boleh hubung Klinik Chong?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-037',
  'language': 'chinese',
  'expected_sections': ['2.4'],
  'query': '怎样联系Klinik Chong',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-038',
  'language': 'chinese',
  'expected_sections': ['2.4'],
  'query': 'Klinik Chong的电话联系号码是?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-039',
  'language': 'malay',
  'expected_sections': ['2.4'],
  'query': 'Tel Number klinik chong ialah?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-040',
  'language': 'chinese',
  'expected_sections': ['2.4'],
  'query': '如果我要询问医生是否有空，可以通过哪些联络渠道？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-041',
  'language': 'malay',
  'expected_sections': ['2.5'],
  'query': 'Klinik Chong bila buka?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-042',
  'language': 'chinese',
  'expected_sections': ['2.5'],
  'query': 'Klinik Chong 几点开?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-043',
  'language': 'malay',
  'expected_sections': ['2.5'],
  'query': 'Bila waktu operasi Klinik chong?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-044',
  'language': 'chinese',
  'expected_sections': ['2.5'],
  'query': '诊所几点关门?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-045',
  'language': 'chinese',
  'expected_sections': ['2.5'],
  'query': '诊所关门后还能使用聊天机器人查询资料吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-046',
  'language': 'chinese',
  'expected_sections': ['3.1'],
  'query': '庄诊所如何保护病人的个人和医疗资料？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-047',
  'language': 'malay',
  'expected_sections': ['3.1'],
  'query': 'Untuk tujuan apakah maklumat sulit pesakit boleh digunakan oleh klinik?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-048',
  'language': 'chinese',
  'expected_sections': ['3.1'],
  'query': '诊所可以在未经同意的情况下把病人资料交给其他人吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-049',
  'language': 'malay',
  'expected_sections': ['3.1'],
  'query': 'Adakah pesakit dibenarkan merakam atau mengambil gambar pesakit lain tanpa kebenaran?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-050',
  'language': 'chinese',
  'expected_sections': ['3.1'],
  'query': '病人在宗教、文化背景和尊严方面享有什么权利？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-051',
  'language': 'chinese',
  'expected_sections': ['3.2'],
  'query': '病人需要向诊所提供哪些准确而完整的健康信息？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-052',
  'language': 'malay',
  'expected_sections': ['3.2'],
  'query': 'Mengapakah pesakit perlu menyemak ringkasan klinikal yang dihasilkan oleh chatbot?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-053',
  'language': 'chinese',
  'expected_sections': ['3.2'],
  'query': '如果机器人的摘要有错误，病人应该怎么处理？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-054',
  'language': 'malay',
  'expected_sections': ['3.2'],
  'query': 'Apakah kesan jika pesakit memberikan maklumat yang tidak lengkap atau tidak tepat?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-055',
  'language': 'chinese',
  'expected_sections': ['3.2'],
  'query': '病人是否应该告知正在服用的药物和已知过敏？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-056',
  'language': 'chinese',
  'expected_sections': ['3.3'],
  'query': '病人应该提前多少分钟抵达诊所？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-057',
  'language': 'malay',
  'expected_sections': ['3.3'],
  'query': 'Apakah yang perlu dilakukan jika pesakit akan tiba lewat?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-058',
  'language': 'chinese',
  'expected_sections': ['3.3'],
  'query': '迟到可能导致预约被怎样处理？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-059',
  'language': 'malay',
  'expected_sections': ['3.3'],
  'query': 'Mengapakah waktu konsultasi kadangkala berubah daripada jadual asal?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-060',
  'language': 'chinese',
  'expected_sections': ['3.3'],
  'query': '如果迟到影响其他病人的时间，诊所可能提供哪些安排？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-061',
  'language': 'chinese',
  'expected_sections': ['3.4'],
  'query': '无法出席预约时，病人有什么责任？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-062',
  'language': 'malay',
  'expected_sections': ['3.4'],
  'query': 'Melalui saluran apakah temu janji boleh dibatalkan atau dijadualkan semula?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-063',
  'language': 'chinese',
  'expected_sections': ['3.4'],
  'query': '取消预约时可能需要提供哪些资料来验证身份？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-064',
  'language': 'malay',
  'expected_sections': ['3.4'],
  'query': 'Berapa awal pesakit digalakkan membatalkan temu janji?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-065',
  'language': 'chinese',
  'expected_sections': ['3.4'],
  'query': '没有收到取消确认时，预约算是已经取消了吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-066',
  'language': 'malay',
  'expected_sections': ['4.1'],
  'query': 'apakah proses lawatan ke klinik chong?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-067',
  'language': 'chinese',
  'expected_sections': ['4.1'],
  'query': '去诊所的流程是什么？',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-068',
  'language': 'chinese',
  'expected_sections': ['4.1'],
  'query': '去诊所做什么?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-069',
  'language': 'malay',
  'expected_sections': ['4.1'],
  'query': 'Apakah pemeriksaan awal yang mungkin dilakukan sebelum konsultasi pertama?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-070',
  'language': 'chinese',
  'expected_sections': ['4.1'],
  'query': '新病人完成医生问诊后可能被安排哪些后续事项？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-071',
  'language': 'chinese',
  'expected_sections': ['4.2'],
  'query': '复诊病人到达柜台后需要确认哪些资料？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-072',
  'language': 'malay',
  'expected_sections': ['4.2'],
  'query': 'Apakah perubahan maklumat yang perlu diberitahu semasa lawatan susulan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-073',
  'language': 'chinese',
  'expected_sections': ['4.2'],
  'query': '复诊时需要携带新的检查结果或治疗文件吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-074',
  'language': 'malay',
  'expected_sections': ['4.2'],
  'query': 'Apakah yang mungkin dinilai oleh doktor semasa konsultasi susulan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-075',
  'language': 'chinese',
  'expected_sections': ['4.2'],
  'query': '医生要求再次复诊时，病人可以在哪里选择新的预约时间？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-076',
  'language': 'malay',
  'expected_sections': ['4.3'],
  'query': 'Kalau pergi klinik nak bawa apa',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-077',
  'language': 'chinese',
  'expected_sections': ['4.3'],
  'query': '如果去诊所要带什么?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-078',
  'language': 'chinese',
  'expected_sections': ['4.3'],
  'query': '如果有以前的化验或影像报告，是否应该带去诊所？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-079',
  'language': 'malay',
  'expected_sections': ['4.3'],
  'query': 'Apakah yang perlu dimasukkan dalam senarai ubat yang dibawa pesakit?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-080',
  'language': 'chinese',
  'expected_sections': ['4.3'],
  'query': '使用保险或公司担保时可能需要准备哪些付款文件？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-081',
  'language': 'chinese',
  'expected_sections': ['4.4'],
  'query': '进入诊疗室前，病人应该准备哪些症状资料？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-082',
  'language': 'malay',
  'expected_sections': ['4.4'],
  'query': 'Bolehkah ringkasan awal chatbot digunakan sebagai rujukan sebelum konsultasi?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-083',
  'language': 'chinese',
  'expected_sections': ['4.4'],
  'query': '等待期间如果身体突然变差，病人应该怎么办？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-084',
  'language': 'malay',
  'expected_sections': ['4.4'],
  'query': 'Apakah yang perlu dilakukan supaya pesakit tidak terlepas nombor giliran?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-085',
  'language': 'chinese',
  'expected_sections': ['4.4'],
  'query': '为什么不同病人的等候时间可能不一样？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-086',
  'language': 'malay',
  'expected_sections': ['4.5'],
  'query': 'semasa konsultasi nak buat apa?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-087',
  'language': 'chinese',
  'expected_sections': ['4.5'],
  'query': '咨询的时候要做什么?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-088',
  'language': 'chinese',
  'expected_sections': ['4.5'],
  'query': '医生在问诊时可能解释哪些检查或治疗事项？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-089',
  'language': 'malay',
  'expected_sections': ['4.5'],
  'query': 'Bolehkah pesakit bertanya jika tidak memahami arahan doktor?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-090',
  'language': 'chinese',
  'expected_sections': ['4.5'],
  'query': '离开诊疗室前，病人应该确认什么？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-091',
  'language': 'chinese',
  'expected_sections': ['4.6'],
  'query': '看诊结束后领取药物时需要检查什么？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-092',
  'language': 'malay',
  'expected_sections': ['4.6'],
  'query': 'Apakah dokumen yang mungkin diterima selepas konsultasi?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-093',
  'language': 'chinese',
  'expected_sections': ['4.6'],
  'query': '需要复诊的话，可以通过哪些方式安排下一次预约？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-094',
  'language': 'malay',
  'expected_sections': ['4.6'],
  'query': 'Bilakah sesuatu temu janji susulan dianggap sah?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-095',
  'language': 'chinese',
  'expected_sections': ['4.6'],
  'query': '回家后症状没有改善或出现新症状时应该怎么办？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-096',
  'language': 'chinese',
  'expected_sections': ['5.1'],
  'query': '如何通过庄诊所聊天机器人预约看诊？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-097',
  'language': 'malay',
  'expected_sections': ['5.1'],
  'query': 'Apakah langkah yang perlu dilakukan sebelum memilih slot melalui chatbot?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-098',
  'language': 'chinese',
  'expected_sections': ['5.1'],
  'query': '除了聊天机器人，还能通过什么方式预约？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-099',
  'language': 'malay',
  'expected_sections': ['5.1'],
  'query': 'Bilakah pesakit akan menerima ID temu janji?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-100',
  'language': 'chinese',
  'expected_sections': ['5.1'],
  'query': '机器人发现预约资料不完整时会怎样处理？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-101',
  'language': 'malay',
  'expected_sections': ['5.2'],
  'query': 'Apakah maklumat peribadi yang diperlukan untuk memproses tempahan temu janji?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-102',
  'language': 'chinese',
  'expected_sections': ['5.2'],
  'query': '预约时是否需要填写出生日期、电话号码和首选语言？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-103',
  'language': 'malay',
  'expected_sections': ['5.2'],
  'query': 'Apakah maklumat gejala yang mungkin diminta sebelum konsultasi?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-104',
  'language': 'chinese',
  'expected_sections': ['5.2'],
  'query': '病人提交预约前为什么要再次检查姓名、日期和时间？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-105',
  'language': 'malay',
  'expected_sections': ['5.2'],
  'query': 'Adakah maklumat gejala yang dikumpulkan menentukan diagnosis atau rawatan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-106',
  'language': 'malay',
  'expected_sections': ['5.3'],
  'query': 'Berapa lama tempoh setiap slot konsultasi dalam prototaip sistem?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-107',
  'language': 'chinese',
  'expected_sections': ['5.3'],
  'query': '系统根据哪些因素决定要显示哪些预约时段？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-108',
  'language': 'malay',
  'expected_sections': ['5.3'],
  'query': 'Bagaimanakah sistem mengelakkan dua pesakit menempah slot yang sama?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-109',
  'language': 'chinese',
  'expected_sections': ['5.3'],
  'query': '如果所选时段已经没有空位，机器人会建议什么替代方案？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-110',
  'language': 'malay',
  'expected_sections': ['5.3'],
  'query': 'Adakah satu slot boleh diberikan kepada lebih daripada seorang pesakit pada masa yang sama?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-111',
  'language': 'malay',
  'expected_sections': ['5.4'],
  'query': 'Bagaimanakah sistem menentukan sama ada seseorang doktor tersedia?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-112',
  'language': 'chinese',
  'expected_sections': ['5.4'],
  'query': '医生状态中的“已预约”和“不开放”分别代表什么？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-113',
  'language': 'malay',
  'expected_sections': ['5.4'],
  'query': 'Bolehkah sistem mencadangkan doktor jika pesakit tidak mempunyai pilihan tertentu?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-114',
  'language': 'chinese',
  'expected_sections': ['5.4'],
  'query': '为什么系统显示的医生可用情况可能随后改变？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-115',
  'language': 'malay',
  'expected_sections': ['5.4'],
  'query': 'Apakah yang berlaku jika perubahan jadual menyebabkan doktor yang dipilih perlu ditukar?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-116',
  'language': 'malay',
  'expected_sections': ['5.5'],
  'query': 'Apakah maklumat yang dipaparkan dalam ringkasan sebelum temu janji disahkan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-117',
  'language': 'chinese',
  'expected_sections': ['5.5'],
  'query': '病人按下确认预约后，系统会执行哪些步骤？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-118',
  'language': 'malay',
  'expected_sections': ['5.5'],
  'query': 'Mengapakah sistem menyemak semula slot sebelum mendaftarkan temu janji?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-119',
  'language': 'chinese',
  'expected_sections': ['5.5'],
  'query': '怎样判断一次预约已经正式成功？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-120',
  'language': 'malay',
  'expected_sections': ['5.5'],
  'query': 'Apakah yang perlu dibuat jika pesakit tidak menerima ID atau mesej pengesahan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-121',
  'language': 'malay',
  'expected_sections': ['5.6'],
  'query': 'Apakah tindakan yang mungkin diambil jika pesakit lewat kurang daripada 15 minit?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-122',
  'language': 'chinese',
  'expected_sections': ['5.6'],
  'query': '迟到15到30分钟的病人可能需要怎样安排？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-123',
  'language': 'malay',
  'expected_sections': ['5.6'],
  'query': 'Adakah temu janji mungkin dijadualkan semula jika kelewatan melebihi 30 minit?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-124',
  'language': 'chinese',
  'expected_sections': ['5.6'],
  'query': '诊所处理迟到情况时会考虑哪些因素？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-125',
  'language': 'malay',
  'expected_sections': ['5.6'],
  'query': 'Adakah klinik menjamin pesakit yang lewat akan diperiksa pada waktu asal?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-126',
  'language': 'malay',
  'expected_sections': ['5.7'],
  'query': 'Apakah langkah untuk membatalkan temu janji melalui chatbot?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-127',
  'language': 'chinese',
  'expected_sections': ['5.7'],
  'query': '取消成功后，原本的预约时段会怎样处理？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-128',
  'language': 'malay',
  'expected_sections': ['5.7'],
  'query': 'Bagaimanakah pesakit memilih tarikh atau waktu baharu untuk penjadualan semula?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-129',
  'language': 'chinese',
  'expected_sections': ['5.7'],
  'query': '如果新的时段还没确认，原来的预约会立即失效吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-130',
  'language': 'malay',
  'expected_sections': ['5.7'],
  'query': 'Apakah maklumat pengesahan yang diperlukan untuk membatalkan atau menjadualkan semula temu '
           'janji?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-131',
  'language': 'malay',
  'expected_sections': ['5.8'],
  'query': 'Adakah Klinik Chong menerima pesakit walk-in semasa waktu operasi?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-132',
  'language': 'chinese',
  'expected_sections': ['5.8'],
  'query': '诊所接收没有预约的病人时会考虑哪些条件？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-133',
  'language': 'malay',
  'expected_sections': ['5.8'],
  'query': 'Mengapakah masa menunggu pesakit walk-in mungkin lebih lama?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-134',
  'language': 'chinese',
  'expected_sections': ['5.8'],
  'query': '如果诊所已满，walk-in病人可能会得到什么安排？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-135',
  'language': 'malay',
  'expected_sections': ['5.8'],
  'query': 'Adakah pesakit yang memerlukan perhatian segera dinilai hanya berdasarkan nombor giliran?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-136',
  'language': 'malay',
  'expected_sections': ['6.1'],
  'query': 'apa yang boleh awak buat?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-137',
  'language': 'malay',
  'expected_sections': ['6.1'],
  'query': 'kamu boleh buat apa?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-138',
  'language': 'malay',
  'expected_sections': ['6.1'],
  'query': 'apakah fungsi kamu?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-139',
  'language': 'chinese',
  'expected_sections': ['6.1'],
  'query': '你能做什么？',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-140',
  'language': 'malay',
  'expected_sections': ['6.1'],
  'query': 'Bolehkah chatbot meminta maklum balas selepas pesakit selesai menggunakan perkhidmatan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-141',
  'language': 'malay',
  'expected_sections': ['6.2'],
  'query': 'kamu boleh terima bahasa apa?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-142',
  'language': 'chinese',
  'expected_sections': ['6.2'],
  'query': '你可以接受什么语言的输入?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-143',
  'language': 'chinese',
  'expected_sections': ['6.2'],
  'query': '你看得懂英文吗?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-144',
  'language': 'chinese',
  'expected_sections': ['6.2'],
  'query': '你看得懂马来文吗？',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-145',
  'language': 'malay',
  'expected_sections': ['6.2'],
  'query': 'Kamu faham bahasa apa?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-146',
  'language': 'malay',
  'expected_sections': ['6.3'],
  'query': 'Apakah maklumat pengenalan diri yang mungkin diminta oleh chatbot?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-147',
  'language': 'chinese',
  'expected_sections': ['6.3'],
  'query': '机器人会收集哪些预约日期、时间和医生资料？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-148',
  'language': 'malay',
  'expected_sections': ['6.3'],
  'query': 'Apakah maklumat berkaitan lawatan dan gejala yang boleh dikumpulkan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-149',
  'language': 'chinese',
  'expected_sections': ['6.3'],
  'query': '系统可能记录哪些与操作有关的技术信息？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-150',
  'language': 'malay',
  'expected_sections': ['6.3'],
  'query': 'Apakah maklumat sensitif yang tidak sepatutnya diberikan kepada chatbot?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-151',
  'language': 'chinese',
  'expected_sections': ['6.4'],
  'query': '你可以看病吗？',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-152',
  'language': 'chinese',
  'expected_sections': ['6.4'],
  'query': '聊天机器人可以开药或更改病人的药物吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-153',
  'language': 'malay',
  'expected_sections': ['6.4'],
  'query': 'Bolehkah chatbot menjamin doktor atau slot tertentu akan kekal tersedia?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-154',
  'language': 'chinese',
  'expected_sections': ['6.4'],
  'query': '哪些因素可能导致机器人的回复不够准确？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-155',
  'language': 'malay',
  'expected_sections': ['6.4'],
  'query': 'Apakah yang mungkin dilakukan oleh chatbot apabila mesej pesakit terlalu ringkas atau tidak '
           'jelas?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-156',
  'language': 'malay',
  'expected_sections': ['6.5'],
  'query': 'Apakah komponen maklumat yang boleh disusun dalam ringkasan klinikal?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-157',
  'language': 'chinese',
  'expected_sections': ['6.5'],
  'query': '保存临床摘要前，机器人会要求病人做什么？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-158',
  'language': 'malay',
  'expected_sections': ['6.5'],
  'query': 'Bolehkah pesakit membetulkan atau menambah maklumat dalam ringkasan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-159',
  'language': 'chinese',
  'expected_sections': ['6.5'],
  'query': '如果病人没有提供某项资料，摘要应该怎样显示？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-160',
  'language': 'malay',
  'expected_sections': ['6.5'],
  'query': 'Adakah ringkasan klinikal dianggap sebagai diagnosis atau arahan rawatan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-161',
  'language': 'malay',
  'expected_sections': ['6.6'],
  'query': 'Apakah yang boleh dilakukan oleh chatbot apabila mengesan kemungkinan kecemasan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-162',
  'language': 'chinese',
  'expected_sections': ['6.6'],
  'query': '发现严重风险时，机器人会继续普通预约流程吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-163',
  'language': 'malay',
  'expected_sections': ['6.6'],
  'query': 'Bolehkah chatbot menghantar ambulans atau menjejaki lokasi pesakit secara automatik?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-164',
  'language': 'chinese',
  'expected_sections': ['6.6'],
  'query': '紧急情况下为什么不应该等待机器人的回复？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-165',
  'language': 'malay',
  'expected_sections': ['6.6'],
  'query': 'Adakah respons kecemasan chatbot menggantikan arahan petugas kecemasan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-166',
  'language': 'malay',
  'expected_sections': ['7.1'],
  'query': 'kalau demam nak buat apa?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-167',
  'language': 'chinese',
  'expected_sections': ['7.1'],
  'query': '发烧时可能同时出现哪些常见症状？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-168',
  'language': 'malay',
  'expected_sections': ['7.1'],
  'query': 'Apakah langkah penjagaan umum untuk demam ringan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-169',
  'language': 'chinese',
  'expected_sections': ['7.1'],
  'query': '发烧持续或反复出现时应该联系诊所吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-170',
  'language': 'malay',
  'expected_sections': ['7.1'],
  'query': 'Siapakah yang dianggap kumpulan berisiko apabila mengalami demam?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-171',
  'language': 'malay',
  'expected_sections': ['7.2'],
  'query': 'boleh bagi nasihat kalau saya batuk.',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-172',
  'language': 'chinese',
  'expected_sections': ['7.2'],
  'query': '咳嗽可以分成哪些常见类型？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-173',
  'language': 'malay',
  'expected_sections': ['7.2'],
  'query': 'Apakah penjagaan umum yang boleh membantu ketika batuk?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-174',
  'language': 'chinese',
  'expected_sections': ['7.2'],
  'query': '出现呼吸道症状时为什么建议戴口罩和经常洗手？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-175',
  'language': 'malay',
  'expected_sections': ['7.2'],
  'query': 'Bilakah batuk perlu dinilai oleh pihak klinik?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-176',
  'language': 'malay',
  'expected_sections': ['7.3'],
  'query': 'Apakah perbezaan umum antara selesema biasa dengan influenza?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-177',
  'language': 'chinese',
  'expected_sections': ['7.3'],
  'query': '流行性感冒可能突然出现哪些症状？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-178',
  'language': 'malay',
  'expected_sections': ['7.3'],
  'query': 'Apakah langkah penjagaan umum apabila mengalami selesema?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-179',
  'language': 'chinese',
  'expected_sections': ['7.3'],
  'query': '感冒症状好转后又再次出现时需要联系诊所吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-180',
  'language': 'malay',
  'expected_sections': ['7.3'],
  'query': 'Adakah influenza boleh berlaku sepanjang tahun di Malaysia?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-181',
  'language': 'malay',
  'expected_sections': ['7.4'],
  'query': 'Kalau saya muntah, apa yang perlu saya buat?',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-182',
  'language': 'malay',
  'expected_sections': ['7.4'],
  'query': 'saya muntah, bagi nasihat kepada saya.',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-183',
  'language': 'malay',
  'expected_sections': ['7.4'],
  'query': 'Apakah tanda-tanda yang mungkin menunjukkan dehidrasi?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-184',
  'language': 'chinese',
  'expected_sections': ['7.4'],
  'query': '一直无法喝下或保留液体时应该联系诊所吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-185',
  'language': 'malay',
  'expected_sections': ['7.4'],
  'query': 'Golongan manakah yang memerlukan perhatian tambahan apabila mengalami muntah?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-186',
  'language': 'malay',
  'expected_sections': ['7.5'],
  'query': 'kalau covid nak buat apa',
  'origin': 'User-provided'},
 {'test_id': 'KC-RAG-187',
  'language': 'chinese',
  'expected_sections': ['7.5'],
  'query': '只根据症状能确定是COVID-19还是流感吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-188',
  'language': 'malay',
  'expected_sections': ['7.5'],
  'query': 'Apakah langkah berjaga-jaga apabila mengalami gejala pernafasan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-189',
  'language': 'chinese',
  'expected_sections': ['7.5'],
  'query': '去诊所前是否应该先告知工作人员自己有呼吸道症状？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-190',
  'language': 'malay',
  'expected_sections': ['7.5'],
  'query': 'Mengapakah penilaian profesional atau ujian mungkin diperlukan untuk gejala pernafasan?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-191',
  'language': 'malay',
  'expected_sections': ['7.6'],
  'query': 'Bilakah gejala yang berterusan atau semakin teruk perlu dibawa kepada perhatian klinik?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-192',
  'language': 'chinese',
  'expected_sections': ['7.6'],
  'query': '哪些普通症状影响日常生活时应该联系庄诊所？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-193',
  'language': 'malay',
  'expected_sections': ['7.6'],
  'query': 'Perlukah pesakit menghubungi klinik jika tidak dapat makan atau minum seperti biasa?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-194',
  'language': 'chinese',
  'expected_sections': ['7.6'],
  'query': '联系诊所讨论症状时应该准备哪些资料？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-195',
  'language': 'malay',
  'expected_sections': ['7.6'],
  'query': 'Siapakah yang boleh menjalankan penilaian klinikal sebenar terhadap pesakit?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-196',
  'language': 'malay',
  'expected_sections': ['7.7'],
  'query': 'Apakah tanda pernafasan yang memerlukan bantuan perubatan segera?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-197',
  'language': 'chinese',
  'expected_sections': ['7.7'],
  'query': '胸口疼痛、突然意识混乱或昏倒时应该怎么办？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-198',
  'language': 'malay',
  'expected_sections': ['7.7'],
  'query': 'Adakah muntah sehingga langsung tidak boleh minum dianggap keadaan yang serius?',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-199',
  'language': 'chinese',
  'expected_sections': ['7.7'],
  'query': '病人情况快速恶化时还应该等待普通预约吗？',
  'origin': 'Generated'},
 {'test_id': 'KC-RAG-200',
  'language': 'malay',
  'expected_sections': ['7.7'],
  'query': 'Mengapakah chatbot tidak boleh mengesahkan bahawa pesakit berada dalam keadaan selamat?',
  'origin': 'Generated'}]


### Evaluate 200 ChromaDB Retrieval Cases
- Retrieves the top three chunks for every question.
- Records the returned section ranking and first relevant rank.
- Prints progress every 25 questions so Run All does not appear frozen.


In [ ]:
if RUN_LOCAL_RETRIEVAL_EVALUATION:
    evaluation_rows = []

    for test_number, test_case in enumerate(retrieval_test_cases, 1):
        results = retrieve_chunks(query=test_case["query"], top_k=3)
        returned_sections = [str(result["section_number"]) for result in results]
        expected_sections = {str(section) for section in test_case["expected_sections"]}

        relevant_rank = next(
            (rank for rank, section in enumerate(returned_sections, 1)
             if section in expected_sections),
            None,
        )
        evaluation_rows.append({
            "test_id": test_case["test_id"],
            "language": test_case["language"],
            "origin": test_case["origin"],
            "query": test_case["query"],
            "expected": ", ".join(sorted(expected_sections)),
            "top_1": returned_sections[0] if returned_sections else None,
            "top_3": ", ".join(returned_sections),
            "relevant_rank": relevant_rank,
            "hit_at_1": relevant_rank == 1,
            "hit_at_3": relevant_rank is not None and relevant_rank <= 3,
            "reciprocal_rank": 1.0 / relevant_rank if relevant_rank else 0.0,
        })

        if test_number % 25 == 0:
            print(f"ChromaDB retrieval evaluation: {test_number}/200 completed")

    evaluation_df = pd.DataFrame(evaluation_rows)
    display(evaluation_df)
else:
    evaluation_candidates = [
        FYP2_ROOT / "RAG_Evaluation_Figures" / "RAG_Retrieval_200_Question_Results.csv",
        FYP2_ROOT / "Evaluation" / "RAG_Evaluation_Figures" / "RAG_Retrieval_200_Question_Results.csv",
    ]
    evaluation_file = next((path for path in evaluation_candidates if path.exists()), None)
    if evaluation_file is None:
        for csv_file in FYP2_ROOT.rglob("RAG_Retrieval_200_Question_Results.csv"):
            evaluation_file = csv_file
            break
    if evaluation_file is None:
        raise FileNotFoundError(
            "Saved RAG_Retrieval_200_Question_Results.csv was not found. "
            "Set RUN_LOCAL_RETRIEVAL_EVALUATION=True to create fresh results."
        )
    evaluation_df = pd.read_csv(evaluation_file, encoding="utf-8-sig")
    print("SKIPPED: 200-query retrieval evaluation; loaded saved verified results:")
    print(evaluation_file)
    display(evaluation_df.head())


### Calculate Hit@1, Hit@3 and MRR
- Reports the overall score and a language-level breakdown.
- Hit@1 checks the first returned section; Hit@3 checks the first three.
- MRR rewards the correct section for appearing earlier in the ranking.


In [ ]:
def retrieval_metric_row(label, frame):
    return {
        "group": label,
        "questions": len(frame),
        "Hit@1": float(frame["hit_at_1"].mean()),
        "Hit@3": float(frame["hit_at_3"].mean()),
        "MRR": float(frame["reciprocal_rank"].mean()),
    }


retrieval_metric_rows = [retrieval_metric_row("Overall", evaluation_df)]
for language, language_frame in evaluation_df.groupby("language", sort=True):
    retrieval_metric_rows.append(retrieval_metric_row(language, language_frame))

retrieval_metrics_df = pd.DataFrame(retrieval_metric_rows)
for metric in ["Hit@1", "Hit@3", "MRR"]:
    retrieval_metrics_df[metric] = retrieval_metrics_df[metric].round(4)

display(retrieval_metrics_df)
print("Overall Hit@1:", retrieval_metrics_df.iloc[0]["Hit@1"])
print("Overall Hit@3:", retrieval_metrics_df.iloc[0]["Hit@3"])
print("Overall MRR:", retrieval_metrics_df.iloc[0]["MRR"])


### ChromaDB Retrieval Metrics Bar Chart
- Compares Hit@1, Hit@3 and MRR for the overall, Chinese and Malay test groups.
- The next setup cell loads every library, Drive path, saved evaluation result, ChromaDB collection and embedding dependency required by all cells below this heading.
- The chart is generated from `retrieval_metrics_df`, which is rebuilt from the saved 200-question evaluation results.


In [ ]:
# RUN_ALL_CONTINUATION_CHECK_V2
# The notebook has already installed packages, mounted Drive, loaded ChromaDB,
# loaded the embedding model and prepared evaluation_df above.
import matplotlib.pyplot as plt

required_runtime_names = [
    "FYP2_ROOT", "DATABASE_DIR", "DB_PATH", "CHROMA_DB_PATH",
    "collection", "embedding_model", "retrieve_chunks", "evaluation_df",
]
missing_runtime_names = [name for name in required_runtime_names if name not in globals()]
if missing_runtime_names:
    raise RuntimeError(
        "Run the notebook from the first cell. Missing runtime value(s): "
        + ", ".join(missing_runtime_names)
    )

print("Run-All continuation check passed ✅")
print("ChromaDB records:", collection.count())
print("Saved retrieval records:", len(evaluation_df))


In [ ]:
import matplotlib.pyplot as plt

plot_metrics = retrieval_metrics_df.copy()

if "group" not in plot_metrics.columns:
    if "Query Language" in plot_metrics.columns:
        plot_metrics = plot_metrics.rename(columns={"Query Language": "group"})
    else:
        raise KeyError(
            "retrieval_metrics_df requires either a 'group' or 'Query Language' column."
        )

plot_metrics["group"] = (
    plot_metrics["group"]
    .astype(str)
    .str.capitalize()
)

plot_metrics = plot_metrics.set_index("group")[[
    "Hit@1",
    "Hit@3",
    "MRR",
]]

ax = plot_metrics.plot(
    kind="bar",
    figsize=(10, 5.5),
    color=["#2563EB", "#10B981", "#F59E0B"],
    width=0.72,
)

ax.set_title(
    "ChromaDB Retrieval Performance",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Test Group")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.08)
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(title="Metric", loc="lower right")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        padding=3,
        fontsize=9,
    )

plt.tight_layout()
plt.show()


## 9. Complete `ask_info` Retrieval Pipeline

**Slot-availability branch**

`User Query → Query Compiler → Slot Intent Gate → SQL Parameter Extractor → SQLite Retrieval → Deterministic Response`

- This branch accepts intent labels `ask_slot_availability` and `check_availability_query`.
- It bypasses the Main Section Analyzer, Subsection Analyzer, ChromaDB, Context Compiler and LLM response generator.
- Doctor names, dates, optional times, holidays, rest days, approved leave and booked slots come only from SQLite.

**Static ask-info branch**

`User Query → Query Compiler → Main Section Analyzer → Subsection Analyzer → ChromaDB → Context Compiler → Emotion-Aware Response Generator`


### Initialize OpenAI Model
- Loads the LLM used by query compilation, analyzers, routing, parameter extraction, context compilation and final response generation.


In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY was not found. Add it to Google Colab Secrets before running this cell."
    )

model = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4.1-mini",
    temperature=0,
)

print("OpenAI model initialized ✅")


### Query Compiler Chain
- Rewrites the current message as a clear standalone query.
- Preserves the doctor name, requested date expression and requested time for slot questions.
- Uses conversation history only when the current message depends on it.
- Does not answer or invent slot information.


In [ ]:
query_compiler_prompt = ChatPromptTemplate.from_template("""
Rewrite the current message into one clear standalone retrieval query.
Keep the query in the user's response language and do not answer it.

For appointment-slot questions:
- preserve the doctor name exactly when stated;
- preserve an explicit date or relative date expression such as hari ini,
  esok, lusa, today, tomorrow, 明天, 后天 or 两天后;
- preserve a requested time when stated;
- make the slot-availability request explicit.
- preserve emoji when stated.
- if it is emotion express sentence state it it is ask with what emotion
- for example: user ask where is the klinik address in a angry tone.

Use history only when the current message depends on earlier context.
Never invent a doctor, date, time, slot or database result.

Language: {response_language}
History: {past_queries}
Current message: {input_query}
""")

query_compiler_chain = query_compiler_prompt | model | StrOutputParser()


#### Query Compiler Test
- Verifies multilingual query reformulation before section analysis.


In [ ]:
if RUN_OPENAI_API_TESTS:
    test_result = query_compiler_chain.invoke({
        "past_queries": "",
        "input_query": "我得了covid-19...怎么办..",
        "response_language": "chinese",
    })
    print("Compiled Query:")
    print(test_result)
else:
    test_result = "我得了 COVID-19，应该怎么办？"
    print("SKIPPED: query-compiler API test; using a fixed downstream sample.")


### Main Section Analyzer Chain
- Selects up to two handbook chapters that are most relevant to the compiled query.
- Uses concise chapter descriptions to reduce ambiguous routing.
- Passes selected chapter numbers to the Subsection Analyzer.


In [ ]:
main_section_prompt = ChatPromptTemplate.from_template("""
Select up to 3 handbook main sections that directly answer the query.

Rules:
- Use only the listed Klinik Chong handbook sections.
- Do not force a section merely because a word is loosely related.
- If the query is unrelated to Klinik Chong, appointment services, or health
  information explicitly covered by the listed sections, return exactly NONE.
- Otherwise return section numbers only, separated by commas.

Query:
{compiled_query}

Main Sections:
{main_sections}
""")

main_section_analyzer_chain = main_section_prompt | model | StrOutputParser()


In [ ]:
main_sections = {
    "1": "Selamat Datang ke Klinik Chong — introduction, supported languages, general disclaimer",
    "2": "Mengenai klinik kami — services, location, contact information, operating hours",
    "3": "Hak dan tanggungjawab pesakit — privacy, accurate information, punctuality, cancellation responsibility",
    "4": "Lawatan Anda ke Klinik — new/follow-up visit process, documents, before/during/after consultation",
    "5": "Panduan temu janji — booking, required information, slots, doctor availability, cancellation/rescheduling, walk-in",
    "6": "Chatbot Klinik Chong — chatbot functions, languages, collected data, limitations, clinical summary",
    "7": "Maklumat kesihatan umum — fever, cough, flu, vomiting, respiratory symptoms, when to seek care"
}

main_sections_text = "\n".join(
    f"{chapter}. {description}"
    for chapter, description in main_sections.items()
)

print("Available Main Sections:")
print(main_sections_text)


#### Main Section Analyzer Test
- Checks that the compiled query is routed to the correct handbook chapter(s).


In [ ]:
if RUN_OPENAI_API_TESTS:
    test_main_section = main_section_analyzer_chain.invoke({
        "compiled_query": test_result,
        "main_sections": main_sections_text,
    })
    print("Selected Main Section:")
    print(test_main_section)
else:
    test_main_section = "8"
    print("SKIPPED: main-section API test; using a fixed downstream sample.")


### Subsection Analyzer Chain
- Selects up to three relevant subsections inside the chosen main chapter(s).
- Limits ChromaDB retrieval to the selected subsection numbers.


In [ ]:
subsection_prompt = ChatPromptTemplate.from_template("""
Select up to 3 subsections that directly answer the query.

Rules:
- Use only the listed Klinik Chong handbook subsections.
- Do not select a subsection based on a weak or indirect match.
- If none of the listed subsections directly supports an answer, return exactly NONE.
- Otherwise return subsection numbers only, separated by commas.

Query:
{compiled_query}

Subsections:
{subsections}
""")

subsection_analyzer_chain = subsection_prompt | model | StrOutputParser()


In [ ]:
def build_subsections_text(selected_main_sections):
    # Build analyzer choices from existing ChromaDB metadata.
    selected = {str(section).strip() for section in selected_main_sections}
    stored = collection.get(include=["metadatas"])
    subsection_map = {}

    for metadata in stored.get("metadatas", []):
        chapter = str(metadata.get("chapter_number", "")).strip()
        section = str(metadata.get("section_number", "")).strip()
        title = str(metadata.get("section_title", "")).strip()
        if chapter in selected and section:
            subsection_map[section] = title

    return "\n".join(
        f"{section}. {subsection_map[section]}"
        for section in sorted(subsection_map)
    )


In [ ]:
selected_main_sections = [
    item.strip() for item in str(test_main_section).split(",")
    if item.strip() and item.strip().upper() != "NONE"
]
subsections_text = build_subsections_text(selected_main_sections)
print("Available Subsections:")
print(subsections_text)


#### Subsection Analyzer Test
- Verifies subsection selection for the compiled query.


In [ ]:
if RUN_OPENAI_API_TESTS:
    test_subsection = subsection_analyzer_chain.invoke({
        "compiled_query": test_result,
        "subsections": subsections_text,
    })
    print("Selected Subsection:")
    print(test_subsection)
else:
    test_subsection = "8.1"
    print("SKIPPED: subsection API test; using a fixed downstream sample.")


### ChromaDB Guided Retrieval
- Parses the selected subsection numbers.
- Retrieves only from analyzer-selected handbook subsections.
- Uses one top result per selected subsection because the handbook is structured approximately as one subsection per chunk.


In [ ]:
def parse_section_numbers(section_output):
    return [
        x.strip()
        for x in section_output.split(",")
        if x.strip()
    ]


def retrieve_selected_subsections(
    compiled_query,
    selected_subsections
):
    subsection_numbers = parse_section_numbers(
        selected_subsections
    )

    retrieved_results = []

    for section_number in subsection_numbers:

        results = retrieve_chunks(
            compiled_query,
            top_k=1,
            where={
                "section_number": section_number
            }
        )

        retrieved_results.extend(results)

    return retrieved_results

#### ChromaDB Guided Retrieval Test
- Confirms that retrieval is restricted to the selected subsection set.


In [ ]:
if RUN_LOCAL_RETRIEVAL_EVALUATION:
    retrieval_candidates = retrieve_selected_subsections(
        compiled_query=test_result,
        selected_subsections=test_subsection,
    )
    print("Retrieved Candidates:", len(retrieval_candidates))
    for result in retrieval_candidates:
        print("=" * 70)
        print("Section:", result["section_number"])
        print("Title:", result["section_title"])
        print(result["document"][:300])
else:
    retrieval_candidates = []
    print("SKIPPED: guided retrieval development example.")


### SQLite Database Connection and Doctor Lookup
- Connects to the persistent Klinik Chong SQLite database.
- Builds a doctor-name-to-ID lookup from active doctors.
- Keeps SQLite as the only factual retrieval source for slot availability.


In [ ]:
DB_PATH = DATABASE_DIR / "klinik_chong.db"
if not DB_PATH.exists():
    raise FileNotFoundError(f"SQLite database not found: {DB_PATH}")

def get_db_connection():
    connection = sqlite3.connect(str(DB_PATH), timeout=30)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON;")
    return connection

def build_doctor_lookup():
    with get_db_connection() as connection:
        rows = connection.execute("""
            SELECT d_id, d_name
            FROM doctor
            WHERE is_active = 1
            ORDER BY d_id;
        """).fetchall()
    return {str(row["d_name"]): str(row["d_id"]) for row in rows}

doctor_lookup = build_doctor_lookup()
doctor_id_to_name = {doctor_id: name for name, doctor_id in doctor_lookup.items()}
print(f"Active doctors loaded from SQLite: {len(doctor_lookup)}")
display(pd.DataFrame(
    [{"doctor_id": doctor_id, "doctor_name": name}
     for name, doctor_id in doctor_lookup.items()]
))


### SQLite Doctor Availability Retrieval
- Retrieves doctors working on a requested date.
- Checks clinic holidays, active doctor schedules and approved doctor leave.
- Returns an explicit clinic-closed result when applicable.


In [ ]:
from datetime import datetime, timedelta


def get_available_doctors(target_date):

    conn = get_db_connection()
    cursor = conn.cursor()

    date_obj = datetime.strptime(
        target_date,
        "%Y-%m-%d"
    )

    day_of_week = date_obj.isoweekday()

    # Check clinic holiday
    cursor.execute("""
        SELECT holiday_name
        FROM clinic_holiday
        WHERE holiday_date = ?
        AND is_closed = 1
    """, (target_date,))

    holiday = cursor.fetchone()

    if holiday:
        conn.close()

        return {
            "date": target_date,
            "clinic_closed": True,
            "reason": holiday[0],
            "doctors": []
        }

    # Get working doctors
    cursor.execute("""
        SELECT
            d.d_id,
            d.d_name,
            d.d_expertise,
            s.start_time,
            s.end_time

        FROM doctor d

        JOIN doctor_weekly_schedule s
        ON d.d_id = s.d_id

        WHERE
            d.is_active = 1
            AND s.day_of_week = ?
            AND s.is_working = 1
    """, (day_of_week,))

    doctors = cursor.fetchall()

    available_doctors = []

    for doctor in doctors:

        d_id = doctor[0]

        # Check approved leave
        cursor.execute("""
            SELECT 1
            FROM doctor_leave
            WHERE d_id = ?
            AND leave_date = ?
            AND status = 'APPROVED'
        """, (
            d_id,
            target_date
        ))

        on_leave = cursor.fetchone()

        if on_leave:
            continue

        available_doctors.append({
            "d_id": doctor[0],
            "d_name": doctor[1],
            "expertise": doctor[2],
            "start_time": doctor[3],
            "end_time": doctor[4]
        })

    conn.close()

    return {
        "date": target_date,
        "clinic_closed": False,
        "doctors": available_doctors
    }

#### Doctor Availability Test
- Verifies dynamic doctor availability against the SQLite schedule data.


In [ ]:
test_doctors = get_available_doctors(
    "2026-09-02"
)

test_doctors

### SQLite Available Slot Retrieval
- Validates the doctor and requested date against SQLite.
- Checks clinic holidays, operating hours, doctor rest days and approved leave.
- Removes lunch-break times, booked slots and elapsed slots for the current Malaysia date.
- Returns an explicit status and reason when no slot is available.


In [ ]:
MALAYSIA_TZ = ZoneInfo("Asia/Kuala_Lumpur")


def get_available_slots(target_date, doctor_id):
    """Return factual slot availability from SQLite only."""
    try:
        target = datetime.strptime(str(target_date), "%Y-%m-%d").date()
    except ValueError:
        return {
            "status": "INVALID_DATE", "date": target_date,
            "doctor_id": doctor_id, "available_slots": [],
            "reason": "Date must use YYYY-MM-DD format.",
        }

    now_my = datetime.now(MALAYSIA_TZ)
    day_of_week = target.isoweekday()

    with get_db_connection() as connection:
        doctor = connection.execute("""
            SELECT d_id, d_name, is_active
            FROM doctor WHERE d_id = ?;
        """, (doctor_id,)).fetchone()
        if not doctor or not doctor["is_active"]:
            return {
                "status": "DOCTOR_NOT_FOUND", "date": target.isoformat(),
                "doctor_id": doctor_id, "available_slots": [],
                "reason": "Doctor was not found or is inactive.",
            }

        base = {
            "date": target.isoformat(),
            "doctor_id": str(doctor["d_id"]),
            "doctor_name": str(doctor["d_name"]),
            "available_slots": [],
        }

        if target < now_my.date():
            return {**base, "status": "PAST_DATE", "reason": "The selected date has already passed."}

        clinic = connection.execute("""
            SELECT appointment_start, last_slot_start, break_start,
                   break_end, slot_duration, is_open
            FROM clinic_schedule WHERE day_of_week = ?;
        """, (day_of_week,)).fetchone()
        if not clinic or not clinic["is_open"]:
            return {**base, "status": "CLINIC_CLOSED", "reason": "The clinic is closed on this day."}

        holiday = connection.execute("""
            SELECT holiday_name FROM clinic_holiday
            WHERE holiday_date = ? AND is_closed = 1;
        """, (target.isoformat(),)).fetchone()
        if holiday:
            return {**base, "status": "HOLIDAY", "reason": f"Clinic holiday: {holiday['holiday_name']}"}

        schedule = connection.execute("""
            SELECT start_time, end_time
            FROM doctor_weekly_schedule
            WHERE d_id = ? AND day_of_week = ? AND is_working = 1;
        """, (doctor_id, day_of_week)).fetchone()
        if not schedule:
            return {**base, "status": "REST_DAY", "reason": "The doctor is not working on this date (rest day)."}

        leave = connection.execute("""
            SELECT 1 AS on_leave
            FROM doctor_leave
            WHERE d_id = ? AND leave_date = ? AND UPPER(status) = 'APPROVED';
        """, (doctor_id, target.isoformat())).fetchone()
        if leave:
            return {**base, "status": "APPROVED_LEAVE", "reason": "Doctor is on approved leave."}

        booked = {
            str(row["start_time"])[:5]
            for row in connection.execute("""
                SELECT start_time FROM appointment
                WHERE d_id = ? AND appointment_date = ?
                  AND UPPER(status) IN ('PENDING', 'CONFIRMED');
            """, (doctor_id, target.isoformat())).fetchall()
        }

    parse_time = lambda value: datetime.strptime(str(value)[:5], "%H:%M")
    start_dt = max(parse_time(clinic["appointment_start"]), parse_time(schedule["start_time"]))
    slot_duration = int(clinic["slot_duration"])
    doctor_last_start = parse_time(schedule["end_time"]) - timedelta(minutes=slot_duration)
    last_dt = min(parse_time(clinic["last_slot_start"]), doctor_last_start)
    break_start = parse_time(clinic["break_start"]) if clinic["break_start"] else None
    break_end = parse_time(clinic["break_end"]) if clinic["break_end"] else None

    slots = []
    current = start_dt
    while current <= last_dt:
        slot = current.strftime("%H:%M")
        in_break = bool(break_start and break_end and break_start <= current < break_end)
        elapsed_today = bool(
            target == now_my.date()
            and datetime.combine(target, current.time(), MALAYSIA_TZ) <= now_my
        )
        if not in_break and not elapsed_today and slot not in booked:
            slots.append(slot)
        current += timedelta(minutes=slot_duration)

    if slots:
        return {**base, "status": "AVAILABLE", "available_slots": slots, "reason": None}

    status = "NO_REMAINING_SLOTS" if target == now_my.date() else "FULLY_BOOKED"
    reason = "All remaining slots for today have passed or are booked." if target == now_my.date() else "All appointment slots are already reserved."
    return {**base, "status": status, "reason": reason}


#### Available Slot Test
- Verifies slot generation and booking exclusion logic.


In [ ]:
test_slots = get_available_slots(
    "2026-09-02",
    "D002"
)

test_slots

### Early Slot-Availability Gate and General Retrieval Router
- Slot requests are detected immediately after Query Compiler.
- `ask_slot_availability` bypasses both section analyzers, ChromaDB and Context Compiler.
- Questions about slot policy or slot duration remain knowledge-base questions.


In [ ]:
SLOT_INTENT_LABELS = {
    "ask_slot_availability", "ask_available_slot", "ask_availability_slot",
    "check_availability_query", "check_slot_availability",
}

STATIC_POLICY_TERMS = re.compile(
    r"(?:迟到|遲到|迟到.*后果|预约.*规则|預約.*規則|"
    r"late|lateness|arrive\s+late|"
    r"lewat|kelewatan|polisi|policy|"
    r"流程|procedure|process|规定|規定)", re.I,
)

SLOT_TERMS = re.compile(
    r"(?:slot(?:s)?|availability|available\s+(?:time|appointment)|"
    r"appointment\s+(?:time|slot)|masa\s+(?:kosong|lapang)|waktu\s+kosong|"
    r"slot\s+kosong|janji\s+temu.*kosong|boleh\s+book|"
    r"可预约|可預約|可以预约|可以預約|预约位|預約位|有位|"
    r"空位|有空|空档|空檔|号源|號源|休息吗|休息嗎)", re.I,
)
SLOT_POLICY_TERMS = re.compile(
    r"(?:berapa\s+lama.*slot|tempoh.*slot|slot.*duration|"
    r"how.*slot.*determined|如何.*决定.*时段|哪些因素.*时段|"
    r"一个slot.*多久|一個slot.*多久)", re.I,
)


def is_slot_availability_query(compiled_query, intention=None):
    normalized_intention = str(intention or "").strip().lower()
    if normalized_intention in SLOT_INTENT_LABELS:
        return True
    text = str(compiled_query or "")
    return bool(SLOT_TERMS.search(text) and not SLOT_POLICY_TERMS.search(text))


retrieval_router_prompt = ChatPromptTemplate.from_template("""
Choose the only retrieval source that can directly answer the query.

Return only one number:
0 = ChromaDB
1 = SQLite
2 = Both
3 = Unsupported or no listed source can answer

Routing rules:

ChromaDB:
- Klinik Chong policies, procedures, services and patient-handbook information
- Basic health information explicitly covered by the handbook

SQLite:
- Current doctors, schedules, approved leave, clinic holidays and availability

Both:
- The answer directly requires handbook information and current database records

Unsupported:
- Cooking, entertainment, finance, technology, shopping, legal advice, or other
  requests unrelated to Klinik Chong's supported clinic and health services
- Any request that cannot be answered from the listed handbook subsections or SQLite

Do not choose the closest topic when the source does not directly answer the query.

Selected handbook subsections:
{selected_subsections}

Query:
{compiled_query}
""")

retrieval_router_chain = retrieval_router_prompt | model | StrOutputParser()


#### Early Slot Gate Test
- Confirms that slot questions with or without doctor/date are captured.
- Confirms that a static slot-duration question remains in the knowledge-base path.


In [ ]:
slot_gate_examples = [
    ("dr Aina ada slot tak", True),
    ("有slot吗?", True),
    ("esok ada slot?", True),
    ("你可以预约dr chong吗?", True),
    ("每个slot的时间是多久？", False),
]
for example, expected in slot_gate_examples:
    actual = is_slot_availability_query(example)
    print("PASS" if actual == expected else "FAIL", actual, "->", example)


### LLM SQL Slot Parameter Extractor
- Uses the LLM first to resolve casual Malay, Chinese, English and mixed-language dates.
- Converts all dates to SQLite `YYYY-MM-DD`; a missing date defaults to the supplied Malaysia reference date.
- Matches one or more partial or misspelled doctor names against active SQLite doctors with a similarity threshold of 0.80.


In [ ]:
MALAYSIA_TZ = ZoneInfo("Asia/Kuala_Lumpur")
DOCTOR_MATCH_THRESHOLD = 0.80


def malaysia_today(reference_date=None):
    if reference_date is None:
        return datetime.now(MALAYSIA_TZ).date()
    if isinstance(reference_date, datetime):
        return reference_date.date()
    if isinstance(reference_date, date):
        return reference_date
    return datetime.strptime(str(reference_date), "%Y-%m-%d").date()


def normalize_doctor_name(value):
    text = re.sub(r"[^a-z0-9 ]+", " ", str(value or "").lower())
    text = re.sub(r"\b(?:dr|doctor|doktor)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def doctor_similarity(mention, full_name):
    mention_normalized = normalize_doctor_name(mention)
    full_normalized = normalize_doctor_name(full_name)
    if not mention_normalized:
        return 0.0

    aliases = {full_normalized, full_normalized.replace(" ", "")}
    tokens = full_normalized.split()
    aliases.update(tokens)
    if len(tokens) > 1:
        aliases.add(" ".join(tokens[-2:]))

    mention_compact = mention_normalized.replace(" ", "")
    if mention_normalized in aliases or mention_compact in aliases:
        return 1.0
    return max(
        SequenceMatcher(None, mention_normalized, alias).ratio()
        for alias in aliases if alias
    )


def match_doctor_mentions(doctor_mentions, doctor_lookup, threshold=DOCTOR_MATCH_THRESHOLD):
    matches_by_id = {}
    for mention in doctor_mentions or []:
        candidate_matches = []
        for full_name, doctor_id in doctor_lookup.items():
            score = doctor_similarity(mention, full_name)
            if score >= threshold:
                candidate_matches.append({
                    "doctor_id": str(doctor_id),
                    "doctor_name": str(full_name),
                    "matched_from": str(mention),
                    "similarity": round(float(score), 4),
                })

        # An exact surname/full-name match takes priority over merely similar
        # names (for example, Chong must not also resolve to Wong).
        exact_matches = [item for item in candidate_matches if item["similarity"] == 1.0]
        if exact_matches:
            candidate_matches = exact_matches

        # Keep every database doctor above 0.80. If two names are similarly
        # close, both remain and both doctors' slots will be returned.
        for match in candidate_matches:
            doctor_id = match["doctor_id"]
            if doctor_id not in matches_by_id or match["similarity"] > matches_by_id[doctor_id]["similarity"]:
                matches_by_id[doctor_id] = match

    return sorted(matches_by_id.values(), key=lambda item: (-item["similarity"], item["doctor_name"]))


In [ ]:
sql_parameter_prompt = ChatPromptTemplate.from_template("""
You extract SQLite parameters for Klinik Chong doctor-slot availability.

Malaysia reference date: {current_date}
Active doctors from SQLite:
{doctor_names}

User query:
{compiled_query}

Return JSON only:
{{"doctor_mentions": [], "target_date": "YYYY-MM-DD", "requested_time": null}}

Rules:
1. Return every doctor mentioned. A surname, given name, joined spelling, typo or
   partial name is allowed. Use the active-doctor list to correct likely typos
   such as `Rag` to `Raj`, but never add an unrelated doctor.
2. If one short doctor name can refer to two active doctors, keep the ambiguous
   short name so both matching doctors can be retrieved.
3. Convert explicit dates such as `9 Sep`, `11 October`, `12-9-2026`, `12/9`,
   `24号` and `1 Jan` into one `YYYY-MM-DD` date.
4. Resolve relative expressions from the Malaysia reference date, including
   hari ini, esok, lusa, 3 hari lagi, minggu depan, Jumaat ini, Selasa ini,
   今天, 明天, 后天, 三天后, 四天后, 五天后, 一个星期后 and 一个月后.
5. For a day/month without a year, use the next occurrence on or after the
   reference date. For `24号`, use the current month unless it has passed.
6. `minggu depan` without a weekday means seven days after the reference date.
   Chinese `下个星期` without a weekday means Monday of the next calendar week.
7. For `Jumaat ini`, `Sabtu ini`, `Selasa ini` or `这个星期三`, choose the next
   upcoming occurrence and never return a past date. `下个星期X` means weekday X
   in the immediately following calendar week.
8. A vague word such as `minggu` without a usable day/date is not a specific
   date; use the reference date. If no date is supplied, use the reference date.
9. If no doctor is supplied, return an empty doctor_mentions list.
10. Convert casual times to `HH:MM` in 24-hour format. A dot may separate hours
    and minutes: `10.00` = `10:00`, `3.30` = `15:30`, `4.25` = `16:25`, and
    `4.30` = `16:30`. In clinic context, bare `pukul 1`, `pukul 2`, `pukul 3`,
    `pukul 4`, Chinese `3点`, and similar 1-5 o'clock expressions mean afternoon.
    Respect explicit am/pm; `12pm` = `12:00`. Use null only when no time appears.
11. Do not round a requested time to a slot. For example, preserve `4.25` as
    `16:25`; SQLite logic will suggest the next two available slots.
12. Do not answer the user and do not wrap the JSON in Markdown.
""")
sql_parameter_chain = sql_parameter_prompt | model | StrOutputParser()


In [ ]:
def extract_sql_parameters(compiled_query, doctor_lookup, reference_date=None):
    """LLM-first extraction followed by SQLite doctor-name fuzzy matching."""
    current_date = malaysia_today(reference_date)
    doctor_names_text = "\n".join(f"- {name}" for name in doctor_lookup)
    raw_output = ""
    parse_error = False
    error_message = None

    try:
        raw_output = sql_parameter_chain.invoke({
            "compiled_query": compiled_query,
            "current_date": current_date.isoformat(),
            "doctor_names": doctor_names_text,
        }).strip()
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_output, flags=re.I)
        data = json.loads(cleaned)
    except Exception as error:
        data = {}
        parse_error = True
        error_message = f"{type(error).__name__}: {error}"

    doctor_mentions = data.get("doctor_mentions") or []
    if isinstance(doctor_mentions, str):
        doctor_mentions = [doctor_mentions]
    doctor_mentions = [str(item).strip() for item in doctor_mentions if str(item).strip()]

    # Recover ambiguous short names directly from the query. For example, the
    # database contains both Chong Zhi Cong and Dr. Chong Yi Ren, so `Dr Chong`
    # must retrieve both even if the LLM canonicalises it to only one full name.
    alias_to_doctors = {}
    for full_name in doctor_lookup:
        for token in set(normalize_doctor_name(full_name).split()):
            if len(token) >= 3:
                alias_to_doctors.setdefault(token, set()).add(full_name)
    query_text = str(compiled_query or "")
    for alias, matching_names in alias_to_doctors.items():
        pattern = rf"\b(?:dr|doctor|doktor)\s*\.?\s*{re.escape(alias)}\b"
        if len(matching_names) > 1 and re.search(pattern, query_text, flags=re.I):
            doctor_mentions.append(alias)
    doctor_mentions = list(dict.fromkeys(doctor_mentions))
    matched_doctors = match_doctor_mentions(doctor_mentions, doctor_lookup)

    target_date = data.get("target_date") or current_date.isoformat()
    try:
        target_date = datetime.strptime(str(target_date), "%Y-%m-%d").date().isoformat()
    except (TypeError, ValueError):
        target_date = current_date.isoformat()
        parse_error = True
        error_message = error_message or "LLM returned an invalid target_date."

    requested_time = data.get("requested_time")
    if requested_time:
        time_match = re.fullmatch(r"([01]?\d|2[0-3]):([0-5]\d)", str(requested_time).strip())
        if time_match:
            requested_time = f"{int(time_match.group(1)):02d}:{time_match.group(2)}"
        else:
            requested_time = None
            parse_error = True
            error_message = error_message or "LLM returned an invalid requested_time."

    return {
        "target_date": target_date,
        "doctor_mentions": doctor_mentions,
        "matched_doctors": matched_doctors,
        "doctor_ids": [item["doctor_id"] for item in matched_doctors],
        "doctor_names": [item["doctor_name"] for item in matched_doctors],
        "requested_time": requested_time,
        "doctor_match_threshold": DOCTOR_MATCH_THRESHOLD,
        "parameter_source": "llm+sqlite_doctor_fuzzy_match",
        "parse_error": parse_error,
        "error": error_message,
        "llm_raw": raw_output,
    }


def _time_to_minutes(value):
    hour, minute = map(int, str(value).split(":"))
    return hour * 60 + minute


def retrieve_slots_from_parameters(parameters, doctor_lookup):
    target_date = parameters["target_date"]
    requested_time = parameters.get("requested_time")
    doctor_ids = parameters.get("doctor_ids") or list(doctor_lookup.values())
    requested_specific_doctor = bool(parameters.get("doctor_ids"))
    doctor_results = []

    for doctor_id in dict.fromkeys(str(item) for item in doctor_ids):
        result = get_available_slots(target_date, doctor_id)
        if requested_time and result.get("status") == "AVAILABLE":
            slots = result.get("available_slots", [])
            available = requested_time in slots
            if available:
                result = {
                    **result,
                    "requested_time": requested_time,
                    "requested_time_available": True,
                    "suggested_slots": [],
                    "available_slots": [requested_time],
                    "status": "AVAILABLE",
                    "reason": None,
                }
            else:
                requested_minutes = _time_to_minutes(requested_time)
                suggested_slots = [
                    slot for slot in slots
                    if _time_to_minutes(slot) > requested_minutes
                ][:2]
                result = {
                    **result,
                    "requested_time": requested_time,
                    "requested_time_available": False,
                    "suggested_slots": suggested_slots,
                    "available_slots": suggested_slots,
                    "status": "NEAREST_SLOTS" if suggested_slots else "REQUESTED_TIME_UNAVAILABLE",
                    "reason": (
                        "The requested time is not a standard or available slot; the next two available slots are returned."
                        if suggested_slots else
                        "The requested time and later slots are unavailable."
                    ),
                }
        doctor_results.append(result)

    return {
        "status": "SLOT_RESULTS",
        "date": target_date,
        "requested_specific_doctor": requested_specific_doctor,
        "requested_time": requested_time,
        "doctors": doctor_results,
    }


#### LLM SQL Parameter Extractor and SQLite Slot Test - 50 Cases
- Uses 40 doctor/date questions and 10 additional explicit-time questions.
- Uses `2026-09-06` as a reproducible Malaysia reference date for relative expressions.
- Evaluates doctor, date and time accuracy, exact match, SQLite execution and nearest-slot suggestions.


In [ ]:
if RUN_SQL_LLM_EVALUATION:
    SQL_TEST_REFERENCE_DATE = date(2026, 9, 6)
    sql_parameter_test_cases = [('dr Aina ada slot tak', ['Dr. Aina Zulaikha'], '2026-09-06', None),
     ('dr lee esok ada slot?', ['Dr. Lee Xin Yi'], '2026-09-07', None),
     ('dr Raj 三天后有预约位子吗?', ['Dr. Raj Kumar'], '2026-09-09', None),
     ('dr Frah 一个星期后有slot吗?', ['Dr. Farah Nadia'], '2026-09-13', None),
     ('有slot吗?', [], '2026-09-06', None),
     ('ada slot?', [], '2026-09-06', None),
     ('esok ada slot?', [], '2026-09-07', None),
     ('13 Sep ada slot?', [], '2026-09-13', None),
     ('14/9 ada slot?', [], '2026-09-14', None),
     ('Jumaat ini ada slot?', [], '2026-09-11', None),
     ('这个星期三有slot吗?', [], '2026-09-09', None),
     ('下个星期三 dr Wong有slot吗?', ['Dr. Wong Kai Jun'], '2026-09-16', None),
     ('dr Nur ada slot Sabtu ini?', ['Dr. Nur Syafiqah'], '2026-09-12', None),
     ('dr Tan 24号 有slot吗', ['Dr. Tan Jia Hao'], '2026-09-24', None),
     ('dr Kavitha 1 October ada slot?', ['Dr. Kavitha Devi'], '2026-10-01', None),
     ('dr Siti ada slot 29/9?', ['Dr. Siti Aisyah'], '2026-09-29', None),
     ('23 October dr Wong ada slot?', ['Dr. Wong Kai Jun'], '2026-10-23', None),
     ('dr chong ada slot minggu ?', ['Chong Zhi Cong', 'Dr. Chong Yi Ren'], '2026-09-06', None),
     ('dr lee ada slot 1 Jan?', ['Dr. Lee Xin Yi'], '2027-01-01', None),
     ('doctor Kavitha ada slot 25/12?', ['Dr. Kavitha Devi'], '2026-12-25', None),
     ('minggu depan boleh book dr Wong?', ['Dr. Wong Kai Jun'], '2026-09-13', None),
     ('dr Lee有位吗下个星期四？', ['Dr. Lee Xin Yi'], '2026-09-10', None),
     ('dr Siti 下个星期二有位吗？', ['Dr. Siti Aisyah'], '2026-09-15', None),
     ('dr Kumar 下个礼拜五有位吗？', ['Dr. Raj Kumar'], '2026-09-11', None),
     ('dr Tan 可以预约吗 明天', ['Dr. Tan Jia Hao'], '2026-09-07', None),
     ('dr Aina boleh book lusa?', ['Dr. Aina Zulaikha'], '2026-09-08', None),
     ('Nur Syafiqah明天休息吗？', ['Dr. Nur Syafiqah'], '2026-09-07', None),
     ('dr chong 17/9 有空吗', ['Chong Zhi Cong', 'Dr. Chong Yi Ren'], '2026-09-17', None),
     ('dr lim 有空吗', ['Dr. Lim Wei Jian'], '2026-09-06', None),
     ('dr Lim ada masa lapang?', ['Dr. Lim Wei Jian'], '2026-09-06', None),
     ('dr chong 4/11 ada slot?', ['Chong Zhi Cong', 'Dr. Chong Yi Ren'], '2026-11-04', None),
     ('我可以预约dr chong吗?', ['Chong Zhi Cong', 'Dr. Chong Yi Ren'], '2026-09-06', None),
     ('Dr Farah ada slot 3 hari', ['Dr. Farah Nadia'], '2026-09-09', None),
     ('Dr farah 四天后有slot吗', ['Dr. Farah Nadia'], '2026-09-10', None),
     ('dr Kavitha 五天后有slot吗', ['Dr. Kavitha Devi'], '2026-09-11', None),
     ('dr Tan dan dr Lee ada slot?', ['Dr. Tan Jia Hao', 'Dr. Lee Xin Yi'], '2026-09-06', None),
     ('dr Nur 和 dr Wong有slot吗 明天', ['Dr. Nur Syafiqah', 'Dr. Wong Kai Jun'], '2026-09-07', None),
     ('一个月后 drAina 和 drSiti有slot吗', ['Dr. Aina Zulaikha', 'Dr. Siti Aisyah'], '2026-10-06', None),
     ('Dr Raj 和 Dr Chong 有slot吗', ['Dr. Raj Kumar', 'Chong Zhi Cong', 'Dr. Chong Yi Ren'], '2026-09-06', None),
     ('Dr Tan ada slot kosong 3 hari lagi?', ['Dr. Tan Jia Hao'], '2026-09-09', None),
     ('dr chong 明天 10.00有slot吗', ['Chong Zhi Cong', 'Dr. Chong Yi Ren'], '2026-09-07', '10:00'),
     ('dr lee 三天后 3.30有slot吗', ['Dr. Lee Xin Yi'], '2026-09-09', '15:30'),
     ('dr Siti 下个星期 4.25 有slot吗', ['Dr. Siti Aisyah'], '2026-09-07', '16:25'),
     ('dr lim ada slot esok pukul 1?', ['Dr. Lim Wei Jian'], '2026-09-07', '13:00'),
     ('dr Farah ada slot pukul 4?', ['Dr. Farah Nadia'], '2026-09-06', '16:00'),
     ('dr Kavitha ada slot pukul 4.30 Jumaat ini?', ['Dr. Kavitha Devi'], '2026-09-11', '16:30'),
     ('dr Kavitha ada slot pukul 2 Selasa ini?', ['Dr. Kavitha Devi'], '2026-09-08', '14:00'),
     ('dr Wong 23/9 3点有slot吗', ['Dr. Wong Kai Jun'], '2026-09-23', '15:00'),
     ('dr Rag ada slot 12/12 3pm？', ['Dr. Raj Kumar'], '2026-12-12', '15:00'),
     ('dr Tan ada slot 23/12 12pm?', ['Dr. Tan Jia Hao'], '2026-12-23', '12:00')]

    sql_test_rows = []
    for test_number, (query, expected_doctors, expected_date, expected_time) in enumerate(sql_parameter_test_cases, 1):
        try:
            parameters = extract_sql_parameters(query, doctor_lookup, reference_date=SQL_TEST_REFERENCE_DATE)
            predicted_doctors = parameters.get("doctor_names", [])
            doctor_match = set(predicted_doctors) == set(expected_doctors)
            date_match = parameters.get("target_date") == expected_date
            predicted_time = parameters.get("requested_time")
            time_match = predicted_time == expected_time
            sqlite_result = retrieve_slots_from_parameters(parameters, doctor_lookup)
            sqlite_statuses = [item.get("status") for item in sqlite_result.get("doctors", [])]
            suggested_slots = [
                {
                    "doctor": item.get("doctor_name"),
                    "requested_time": item.get("requested_time"),
                    "suggested_slots": item.get("suggested_slots", []),
                }
                for item in sqlite_result.get("doctors", [])
                if item.get("suggested_slots")
            ]
            sqlite_executed = bool(sqlite_result.get("doctors"))
            error = parameters.get("error")
        except Exception as exception:
            parameters = {}
            predicted_doctors = []
            predicted_time = None
            doctor_match = date_match = time_match = sqlite_executed = False
            sqlite_statuses = []
            suggested_slots = []
            error = f"{type(exception).__name__}: {exception}"

        exact_match = doctor_match and date_match and time_match
        sql_test_rows.append({
            "test": test_number,
            "query": query,
            "expected_doctors": ", ".join(expected_doctors) if expected_doctors else "ALL ACTIVE DOCTORS",
            "predicted_doctors": ", ".join(predicted_doctors) if predicted_doctors else "ALL ACTIVE DOCTORS",
            "expected_date": expected_date,
            "predicted_date": parameters.get("target_date"),
            "expected_time": expected_time,
            "predicted_time": predicted_time,
            "doctor_correct": doctor_match,
            "date_correct": date_match,
            "time_correct": time_match,
            "exact_match": exact_match,
            "sqlite_executed": sqlite_executed,
            "sqlite_statuses": ", ".join(str(item) for item in sqlite_statuses),
            "suggested_slots": json.dumps(suggested_slots, ensure_ascii=False),
            "error": error,
        })
        print(f"SQL extraction test {test_number}/50:", "PASS" if exact_match else "FAIL")

    sql_parameter_test_df = pd.DataFrame(sql_test_rows)
    display(sql_parameter_test_df)

    time_test_df = sql_parameter_test_df[sql_parameter_test_df["expected_time"].notna()]
    sql_extraction_metrics_df = pd.DataFrame([
        {"metric": "Doctor Accuracy", "score": sql_parameter_test_df["doctor_correct"].mean(), "cases": 50},
        {"metric": "Date Accuracy", "score": sql_parameter_test_df["date_correct"].mean(), "cases": 50},
        {"metric": "Time Accuracy", "score": time_test_df["time_correct"].mean(), "cases": len(time_test_df)},
        {"metric": "Exact Match", "score": sql_parameter_test_df["exact_match"].mean(), "cases": 50},
        {"metric": "SQLite Execution Accuracy", "score": sql_parameter_test_df["sqlite_executed"].mean(), "cases": 50},
    ])
    sql_extraction_metrics_df["score"] = sql_extraction_metrics_df["score"].round(4)
    display(sql_extraction_metrics_df)
    print(f"Time Accuracy is evaluated on {len(time_test_df)} explicit-time questions (tests 41-50).")
else:
    sql_parameter_test_df = pd.DataFrame()
    sql_extraction_metrics_df = pd.DataFrame([
        {"metric": "Doctor Accuracy", "score": 1.00, "cases": 50},
        {"metric": "Date Accuracy", "score": 0.98, "cases": 50},
        {"metric": "Time Accuracy", "score": 1.00, "cases": 10},
        {"metric": "Exact Match", "score": 0.98, "cases": 50},
        {"metric": "SQLite Execution Accuracy", "score": 1.00, "cases": 50},
    ])
    print(
        "SKIPPED: 50-case SQL LLM evaluation; loaded metrics from the last "
        "completed run. No OpenAI API call was made."
    )
    display(sql_extraction_metrics_df)


### SQL Extraction and Retrieval Metrics Bar Chart
- Visualises doctor, date, time, exact-match and SQLite execution accuracy.
- Reads the latest values from `sql_extraction_metrics_df`.


In [ ]:
import matplotlib.pyplot as plt
sql_plot_df = sql_extraction_metrics_df.dropna(subset=["score"]).copy()
sql_plot_df["score"] = pd.to_numeric(sql_plot_df["score"])

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(
    sql_plot_df["metric"],
    sql_plot_df["score"],
    color=["#2563EB", "#10B981", "#8B5CF6", "#F59E0B", "#0EA5E9"],
)
ax.set_title("SQL Parameter Extraction and SQLite Retrieval Performance", fontsize=14, fontweight="bold")
ax.set_xlabel("Evaluation Metric")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.08)
ax.tick_params(axis="x", rotation=18)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.bar_label(bars, labels=[f"{score:.3f}" for score in sql_plot_df["score"]], padding=3)
plt.tight_layout()
plt.show()


### SQL-Only Slot Retrieval and General Router
- Uses every matched doctor ID; two matched doctors produce two slot-result records.
- A missing doctor means all active doctors are checked for the selected/default date.
- Slot requests never call ChromaDB, Main Section Analyzer, Subsection Analyzer or Context Compiler.


In [ ]:
def run_sql_slot_retrieval(compiled_query, doctor_lookup, reference_date=None):
    parameters = extract_sql_parameters(
        compiled_query, doctor_lookup, reference_date=reference_date,
    )
    sqlite_result = retrieve_slots_from_parameters(parameters, doctor_lookup)
    return {
        "route": "1",
        "branch": "ask_slot_availability_sql_only",
        "sql_params": parameters,
        "chroma": [],
        "sqlite": {"slot_availability": sqlite_result},
    }


def routed_retrieval(compiled_query, selected_subsections, doctor_lookup):
    if is_slot_availability_query(compiled_query):
        return run_sql_slot_retrieval(compiled_query, doctor_lookup)

    retrieval_route = retrieval_router_chain.invoke({
        "compiled_query": compiled_query,
        "selected_subsections": selected_subsections or "",
    }).strip()
    if retrieval_route not in {"0", "1", "2", "3"}:
        retrieval_route = "3"

    if retrieval_route == "3":
        return {
            "route": "3",
            "branch": "unsupported_or_no_context",
            "sql_params": None,
            "chroma": [],
            "sqlite": {},
        }

    chroma_results, sqlite_results = [], {}
    if retrieval_route in {"0", "2"}:
        chroma_results = retrieve_selected_subsections(
            compiled_query=compiled_query,
            selected_subsections=selected_subsections,
        )
    if retrieval_route in {"1", "2"}:
        sqlite_results["available_doctors_today"] = get_available_doctors(
            malaysia_today().isoformat()
        )

    return {
        "route": retrieval_route,
        "branch": "general_ask_info",
        "sql_params": None,
        "chroma": chroma_results,
        "sqlite": sqlite_results,
    }


#### Routed Retrieval Tests
- Validates ChromaDB-only, SQLite-only and combined retrieval branches.


In [ ]:
if RUN_OPENAI_API_TESTS:
    test_rag = routed_retrieval(
        compiled_query="去Klinik Chong 要带什么？",
        selected_subsections="5.6",
        doctor_lookup=doctor_lookup,
    )
    print("Static query route:", test_rag["route"])
else:
    test_rag = None
    print("SKIPPED: routed-retrieval API development test.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    test_sql = routed_retrieval(
        compiled_query="Dr Chong 下星期五有什么slot？",
        selected_subsections="",
        doctor_lookup=doctor_lookup,
    )
    print("Slot query branch:", test_sql["branch"])
    print("Chroma results:", len(test_sql["chroma"]))
else:
    test_sql = None
    print("SKIPPED: routed-retrieval API development test.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    test_slot_priority = routed_retrieval(
        compiled_query="Dr Chong 明天有空吗？如果迟到会怎样？",
        selected_subsections="5.6",
        doctor_lookup=doctor_lookup,
    )
    print("Mixed slot query still uses:", test_slot_priority["branch"])
    print("Chroma bypassed:", test_slot_priority["chroma"] == [])
else:
    test_slot_priority = None
    print("SKIPPED: routed-retrieval API development test.")


### Context Compiler Chain
- Combines retrieved ChromaDB and SQLite information.
- Keeps only information needed to answer the compiled query.
- Removes duplicated or irrelevant details.
- Does not introduce information that is absent from the retrieved context.


In [ ]:
context_compiler_prompt = ChatPromptTemplate.from_template("""
Compile only the retrieved information that directly answers the query.

Rules:
- Use only the retrieved Klinik Chong handbook or SQLite information below.
- Do not use general knowledge or add medical, cooking, lifestyle or other advice.
- Remove duplicated and irrelevant details.
- If the retrieved information does not directly answer the query, return exactly
  NO_RELEVANT_CONTEXT.
- Do not answer the user directly.

Query:
{compiled_query}

Retrieved information:
{retrieved_context}
""")

context_compiler_chain = context_compiler_prompt | model | StrOutputParser()


In [ ]:
def format_retrieval_results(retrieval_results):
    parts = []
    chroma_results = retrieval_results.get("chroma", [])
    if chroma_results:
        parts.append("ChromaDB:")
        for result in chroma_results:
            parts.append(
                f'Section {result["section_number"]} - {result["section_title"]}\n'
                f'{result["document"]}'
            )

    sqlite_results = retrieval_results.get("sqlite", {})
    if sqlite_results:
        parts.append("\nSQLite:")
        for key, value in sqlite_results.items():
            parts.append(f"{key}: {value}")
    return "\n\n".join(parts)


def has_relevant_context(retrieval_results, min_similarity=MIN_RAG_SIMILARITY):
    if retrieval_results.get("route") == "3":
        return False
    if retrieval_results.get("sqlite"):
        return True
    similarities = [
        float(item.get("similarity", 0.0))
        for item in retrieval_results.get("chroma", [])
    ]
    return bool(similarities and max(similarities) >= float(min_similarity))


def no_relevant_information_response(response_language):
    language = str(response_language or "").strip().lower()
    if language in {"zh", "chinese", "中文", "simplified chinese"}:
        return (
            "很抱歉，Klinik Chong 的资料库中没有找到相关资料。"
            "此聊天机器人只提供诊所服务、预约，以及资料库涵盖的基本健康信息。"
        )
    return (
        "Maaf, maklumat berkaitan tidak ditemui dalam pangkalan pengetahuan "
        "Klinik Chong. Chatbot ini hanya menyediakan maklumat tentang "
        "perkhidmatan klinik, janji temu dan maklumat kesihatan asas yang disokong."
    )


#### Context Compiler Test
- Formats routed retrieval results and verifies that the compiler produces concise grounded context.


In [ ]:
if RUN_OPENAI_API_TESTS:
    retrieved_context_text = format_retrieval_results(test_rag)
    compiled_context = context_compiler_chain.invoke({
        "compiled_query": "Klinik Chong 的迟到政策是什么？",
        "retrieved_context": retrieved_context_text,
    })
    print("Retrieved Context:")
    print(retrieved_context_text)
    print("\nCompiled Context:")
    print(compiled_context)
else:
    retrieved_context_text = ""
    compiled_context = "NO_RELEVANT_CONTEXT"
    print("SKIPPED: context-compiler API development test.")


### Emotion-Aware Response Generator
- Detects one of six emotion labels: `neutral`, `happy`, `sad`, `angry`, `afraid`, `confuse`.
- Uses the emotion only to adjust response tone.
- Answers only from the compiled context.
- Responds in the detected response language without exposing the emotion label to the user-facing response.


In [ ]:
emotion_response_prompt = ChatPromptTemplate.from_template("""
Classify the user's emotion into exactly one label:
neutral, happy, sad, angry, afraid, confuse.

Generate a natural, human-like response according to the rules below.

GROUNDING AND SCOPE RULES:
- Use the detected emotion only to adjust tone and presentation.
- Do not mention or reveal the detected emotion.
- Answer only using facts in the provided Klinik Chong context.
- Do not provide consultation or advice unrelated to Klinik Chong's medical and
  clinic services.
- Do not answer beyond the knowledge-based RAG or SQLite context.
- Do not invent clinic information, medical advice, appointment details or actions.
- If the context is empty, irrelevant, insufficient, or equals NO_RELEVANT_CONTEXT,
  state only that no relevant information was found in the Klinik Chong knowledge
  base. Do not add general knowledge, suggestions, alternatives, clarification
  questions, or offers of further help.
- Preserve important factual details from valid context.
- Respond in {response_language}.
- Keep the response concise, polite and conversational.
- Do not make medical diagnoses or provide unsupported reassurance.
- Emergency instructions present in the context remain direct and prioritised.

TONE RULES:
- neutral: Respond normally in a clear and polite manner.
- happy: Respond warmly and positively without excessive excitement.
- sad: Briefly express understanding, then provide supported information gently.
- angry: Remain calm, respectful and non-defensive; do not blame or lecture.
- afraid: Use a gentle tone without dismissing risk or promising that all is well.
- confuse: Organise supported information into short numbered steps only when the
  context contains actionable steps.

Return exactly:
Emotion: <one emotion label>
Response: <final response>

User:
{user_input}

Context:
{compiled_context}
""")

emotion_response_chain = emotion_response_prompt | model | StrOutputParser()


#### Emotion-Aware Response Test
- Checks grounded response generation and language control.


In [ ]:
# This deterministic test verifies that empty retrieval never reaches generation.
empty_retrieval_test = {
    "route": "3", "branch": "unsupported_or_no_context",
    "sql_params": None, "chroma": [], "sqlite": {},
}
assert has_relevant_context(empty_retrieval_test) is False
assert "没有找到相关资料" in no_relevant_information_response("chinese")
print("PASS: unsupported/empty retrieval uses a deterministic grounded response.")

if RUN_OPENAI_API_TESTS:
    final_response = emotion_response_chain.invoke({
        "user_input": "Klinik Chong 在哪里？",
        "compiled_context": compiled_context,
        "response_language": "chinese",
    })
    print(final_response)
else:
    final_response = None
    print("SKIPPED: emotion-response API development test.")


### Combined `ask_info` Pipeline
- Compiles every query into a standalone form.
- Immediately diverts slot availability to the SQL-only branch.
- Executes Main/Subsection analysis and ChromaDB retrieval only for non-slot information.
- Returns intermediate routing information for verification.


In [ ]:
def parse_emotion_response(output):

    emotion = "neutral"
    response_lines = []
    response_started = False

    for raw_line in str(output).splitlines():

        line = raw_line.strip()
        clean_line = line.replace("**", "").strip()

        if clean_line.lower().startswith("emotion:"):

            detected_emotion = clean_line.split(":", 1)[1].strip()

            if detected_emotion:
                emotion = detected_emotion.lower()

        elif clean_line.lower().startswith("response:"):

            response_started = True
            first_response_line = clean_line.split(":", 1)[1].strip()

            if first_response_line:
                response_lines.append(first_response_line)

        elif response_started and line:

            response_lines.append(line)

    response = "\n".join(response_lines).strip()

    # Fallback if the model did not follow the exact format
    if not response:

        remaining_lines = []

        for raw_line in str(output).splitlines():

            clean_line = raw_line.replace("**", "").strip()

            if not clean_line.lower().startswith("emotion:"):
                if not clean_line.lower().startswith("response:"):
                    if clean_line:
                        remaining_lines.append(clean_line)

        response = "\n".join(remaining_lines).strip()

    return {
        "emotion": emotion,
        "response": response
    }

In [ ]:
def format_slot_response(retrieval, response_language):
    result = retrieval["sqlite"]["slot_availability"]
    language = str(response_language or "").lower()
    is_chinese = language in {"zh", "chinese", "中文"}
    target_date = result.get("date", "")
    doctor_results = result.get("doctors", [])

    if not doctor_results:
        return (
            f"{target_date} 暂时找不到可查询的医生预约资料。" if is_chinese
            else f"Tiada maklumat slot doktor yang dapat disemak pada {target_date}."
        )

    lines = []
    for doctor_result in doctor_results:
        doctor_name = doctor_result.get("doctor_name") or doctor_result.get("doctor_id") or "Doctor"
        status = doctor_result.get("status")
        slots = doctor_result.get("available_slots", [])
        if status == "AVAILABLE" and slots:
            slot_text = ", ".join(slots)
            lines.append(
                f"{doctor_name} 在 {target_date} 可预约的时间是：{slot_text}。" if is_chinese
                else f"Slot {doctor_name} yang tersedia pada {target_date}: {slot_text}."
            )
        elif status == "NEAREST_SLOTS" and slots:
            slot_text = ", ".join(slots)
            requested_time = doctor_result.get("requested_time", "")
            lines.append(
                f"{doctor_name} 在 {target_date} 的 {requested_time} 不可预约；之后最接近的时间是：{slot_text}。" if is_chinese
                else f"Slot {requested_time} untuk {doctor_name} pada {target_date} tidak tersedia; slot terdekat selepas itu: {slot_text}."
            )
        else:
            reason = doctor_result.get("reason") or status or "No slot is available."
            lines.append(
                f"{doctor_name} 在 {target_date} 没有可预约时间，原因：{reason}" if is_chinese
                else f"{doctor_name} tiada slot pada {target_date}. Sebab: {reason}"
            )
    return "\n".join(lines)


def _selected_section_list(raw_selection):
    selection = str(raw_selection or "").strip()
    if not selection or "NONE" in selection.upper():
        return []
    return [item.strip() for item in selection.split(",") if item.strip()]


def _unsupported_result(compiled_query, response_language, main_sections=None, subsections=None):
    response = no_relevant_information_response(response_language)
    return {
        "compiled_query": compiled_query,
        "main_sections": main_sections,
        "subsections": subsections,
        "retrieval": {
            "route": "3", "branch": "unsupported_or_no_context",
            "sql_params": None, "chroma": [], "sqlite": {},
        },
        "compiled_context": "NO_RELEVANT_CONTEXT",
        "emotion": "neutral",
        "response": response,
    }


def run_rag_pipeline(user_input, response_language, past_queries="", intention=None):
    compiled_query = query_compiler_chain.invoke({
        "past_queries": past_queries,
        "input_query": user_input,
        "response_language": response_language,
    }).strip()

    if is_slot_availability_query(compiled_query, intention=intention):
        retrieval = run_sql_slot_retrieval(compiled_query, doctor_lookup)
        response = format_slot_response(retrieval, response_language)
        return {
            "compiled_query": compiled_query,
            "main_sections": None,
            "subsections": None,
            "retrieval": retrieval,
            "compiled_context": json.dumps(retrieval["sqlite"], ensure_ascii=False),
            "emotion": "neutral",
            "response": response,
        }

    selected_main_sections = main_section_analyzer_chain.invoke({
        "compiled_query": compiled_query,
        "main_sections": main_sections_text,
    }).strip()
    main_section_list = _selected_section_list(selected_main_sections)
    if not main_section_list:
        return _unsupported_result(
            compiled_query, response_language, selected_main_sections, None
        )

    available_subsections = build_subsections_text(main_section_list)
    selected_subsections = subsection_analyzer_chain.invoke({
        "compiled_query": compiled_query,
        "subsections": available_subsections,
    }).strip()
    if not _selected_section_list(selected_subsections):
        return _unsupported_result(
            compiled_query, response_language, selected_main_sections, selected_subsections
        )

    retrieval = routed_retrieval(compiled_query, selected_subsections, doctor_lookup)
    if not has_relevant_context(retrieval):
        return _unsupported_result(
            compiled_query, response_language, selected_main_sections, selected_subsections
        )

    retrieved_context = format_retrieval_results(retrieval)
    compiled_context = context_compiler_chain.invoke({
        "compiled_query": compiled_query,
        "retrieved_context": retrieved_context,
    }).strip()
    if not compiled_context or "NO_RELEVANT_CONTEXT" in compiled_context.upper():
        return _unsupported_result(
            compiled_query, response_language, selected_main_sections, selected_subsections
        )

    raw_output = emotion_response_chain.invoke({
        "user_input": user_input,
        "compiled_context": compiled_context,
        "response_language": response_language,
    })
    final_output = parse_emotion_response(raw_output)
    return {
        "compiled_query": compiled_query,
        "main_sections": selected_main_sections,
        "subsections": selected_subsections,
        "retrieval": retrieval,
        "compiled_context": compiled_context,
        "emotion": final_output["emotion"],
        "response": final_output["response"],
    }


## 10. End-to-End Development Tests
- Verifies static ChromaDB questions and SQL-only slot questions.
- Slot tests must return `ask_slot_availability_sql_only` with an empty Chroma result.
- The 200-question retrieval evaluation and 40-question SQL extraction evaluation run earlier in the notebook.


In [ ]:
def print_rag_result(title, result):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    print("Compiled Query:")
    print(result.get("compiled_query"))

    print("\nMain Sections:")
    print(result.get("main_sections"))

    print("\nSubsections:")
    print(result.get("subsections"))

    retrieval = result.get("retrieval", {})

    print("\nRoute:")
    print(retrieval.get("route"))

    print("\nSQL Params:")
    print(retrieval.get("sql_params"))

    print("\nSQLite:")
    print(retrieval.get("sqlite"))

    print("\nCompiled Context:")
    print(result.get("compiled_context"))

    print("\nEmotion:")
    print(result.get("emotion"))

    print("\nFinal Response:")
    print(result.get("response"))

In [ ]:
if RUN_OPENAI_API_TESTS:
    result_1 = run_rag_pipeline(
        user_input="Klinik Chong 几点开门",
        response_language="chinese",
    )
    print_rag_result("TEST 1 - RAG ONLY: Clinic Operating Hours", result_1)
else:
    print("SKIPPED: end-to-end OpenAI API test 1/6.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    result_2 = run_rag_pipeline(
        user_input="如果我预约迟到了会怎样？",
        response_language="chinese",
    )
    print_rag_result("TEST 2 - RAG ONLY: Late Policy", result_2)
else:
    print("SKIPPED: end-to-end OpenAI API test 2/6.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    result_3 = run_rag_pipeline(
        user_input="Dr Lim！我想要预约明天可以吗？",
        response_language="chinese",
        intention="ask_slot_availability",
    )
    print_rag_result("TEST 3 - SQLITE ONLY: Doctor Slots", result_3)
else:
    print("SKIPPED: end-to-end OpenAI API test 3/6.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    result_4 = run_rag_pipeline(
        user_input="我一直呕吐..怎么办",
        response_language="chinese",
        intention="ask_info",
    )
    print_rag_result("TEST 4 - SQLITE ONLY: Available Doctors", result_4)
else:
    print("SKIPPED: end-to-end OpenAI API test 4/6.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    result_5 = run_rag_pipeline(
        user_input="这真的不明白这个chatbot能做什么",
        response_language="malay",
        intention="ask_info",
    )
    print_rag_result("TEST 5 - RAG ONLY: Chatbot Functions", result_5)
else:
    print("SKIPPED: end-to-end OpenAI API test 5/6.")


In [ ]:
if RUN_OPENAI_API_TESTS:
    result_6 = run_rag_pipeline(
        user_input="apa yang perlu saya buat jika appointment lepas masa",
        response_language="malay",
        intention="ask_info",
    )
    print_rag_result("TEST 6 - RAG ONLY: Malay Late-Arrival Policy", result_6)
else:
    print("SKIPPED: end-to-end OpenAI API test 6/6.")


In [ ]:
# ============================================================
# Calculate final metrics and generate Figure 6.1.4.1
# Continue directly from the existing evaluation_df
# ============================================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Confirm that the 200-question results exist
if "evaluation_df" not in globals():
    raise NameError(
        "evaluation_df was not found. "
        "Run the 200-question evaluation first."
    )

if len(evaluation_df) != 200:
    raise ValueError(
        f"Expected 200 results, but found {len(evaluation_df)}."
    )


# ============================================================
# 1. Calculate Hit@1, Hit@3 and MRR
# ============================================================

def calculate_metrics(group_name, dataframe):
    return {
        "Query Language": group_name,
        "Questions": len(dataframe),
        "Hit@1": float(dataframe["hit_at_1"].mean()),
        "Hit@3": float(dataframe["hit_at_3"].mean()),
        "MRR": float(
            dataframe["reciprocal_rank"].mean()
        )
    }


metric_rows = []

for language in ["chinese", "malay"]:

    language_df = evaluation_df[
        evaluation_df["language"] == language
    ]

    metric_rows.append(
        calculate_metrics(
            language.capitalize(),
            language_df
        )
    )


metric_rows.append(
    calculate_metrics(
        "Overall",
        evaluation_df
    )
)


retrieval_metrics_df = pd.DataFrame(
    metric_rows
)


for metric in ["Hit@1", "Hit@3", "MRR"]:

    retrieval_metrics_df[metric] = (
        retrieval_metrics_df[metric].round(4)
    )


print("Final RAG retrieval metrics:")
display(retrieval_metrics_df)


# ============================================================
# 2. Prepare chart values
# ============================================================

query_groups = retrieval_metrics_df[
    "Query Language"
].tolist()

hit_at_1_values = retrieval_metrics_df[
    "Hit@1"
].to_numpy(dtype=float)

hit_at_3_values = retrieval_metrics_df[
    "Hit@3"
].to_numpy(dtype=float)

mrr_values = retrieval_metrics_df[
    "MRR"
].to_numpy(dtype=float)


# ============================================================
# 3. Generate grouped bar chart
# ============================================================

x = np.arange(len(query_groups))
width = 0.23

fig, ax = plt.subplots(
    figsize=(11, 6.5)
)


bars_hit_1 = ax.bar(
    x - width,
    hit_at_1_values,
    width,
    label="Hit@1",
    color="#4C78A8"
)

bars_hit_3 = ax.bar(
    x,
    hit_at_3_values,
    width,
    label="Hit@3",
    color="#59A14F"
)

bars_mrr = ax.bar(
    x + width,
    mrr_values,
    width,
    label="MRR",
    color="#F28E2B"
)


# Display exact values above the bars
for bars in [
    bars_hit_1,
    bars_hit_3,
    bars_mrr
]:
    for bar in bars:

        value = bar.get_height()

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            value + 0.015,

            f"{value:.4f}",

            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )


# ============================================================
# 4. Format chart
# ============================================================

ax.set_title(
    "RAG Retrieval Performance by Query Language",
    fontsize=16,
    fontweight="bold",
    pad=16
)

ax.set_xlabel(
    "Query Language",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylabel(
    "Score",
    fontsize=12,
    fontweight="bold"
)

ax.set_xticks(x)
ax.set_xticklabels(query_groups)

ax.set_ylim(0, 1.05)

ax.set_yticks(
    np.arange(0, 1.01, 0.1)
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

ax.legend(
    title="Retrieval Metric",
    loc="lower right"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()


# ============================================================
# 5. Resolve output location
# ============================================================

if (
    "FYP2_ROOT" in globals()
    and Path(FYP2_ROOT).exists()
):
    output_root = Path(FYP2_ROOT)

elif Path(
    "/content/drive/MyDrive/FYP2"
).exists():
    output_root = Path(
        "/content/drive/MyDrive/FYP2"
    )

elif Path(
    "/content/gdrive/MyDrive/FYP2"
).exists():
    output_root = Path(
        "/content/gdrive/MyDrive/FYP2"
    )

else:
    output_root = Path("/content/FYP2")


output_folder = (
    output_root
    / "RAG_Evaluation_Figures"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


figure_path = (
    output_folder
    / "Figure_6.1.4.1_RAG_Retrieval_Performance.png"
)

metrics_path = (
    output_folder
    / "RAG_Retrieval_Final_Metrics.csv"
)

results_path = (
    output_folder
    / "RAG_Retrieval_200_Question_Results.csv"
)


# ============================================================
# 6. Save outputs
# ============================================================

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

retrieval_metrics_df.to_csv(
    metrics_path,
    index=False,
    encoding="utf-8-sig"
)

evaluation_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

plt.show()


print("\nOutputs saved successfully.")

print("\nFigure:")
print(figure_path)

print("\nMetrics:")
print(metrics_path)

print("\n200-question results:")
print(results_path)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ============================================================
# 1. Load evaluation results
# ============================================================

# Use evaluation_df if it already exists.
# Otherwise, search Google Drive for the saved evaluation CSV.
if "evaluation_df" not in globals():
    search_roots = [
        Path("/content/drive/MyDrive/FYP2"),
        Path("/content/gdrive/MyDrive/FYP2"),
    ]

    evaluation_file = None

    for root in search_roots:
        if not root.exists():
            continue

        for csv_file in root.rglob("*.csv"):
            try:
                columns = pd.read_csv(csv_file, nrows=0).columns

                if {
                    "test_id",
                    "language",
                    "relevant_rank"
                }.issubset(columns):
                    evaluation_file = csv_file
                    break

            except Exception:
                pass

        if evaluation_file is not None:
            break

    if evaluation_file is None:
        raise FileNotFoundError(
            "Could not find a RAG evaluation CSV containing "
            "test_id, language and relevant_rank."
        )

    evaluation_df = pd.read_csv(evaluation_file)

    print("Loaded evaluation results from:")
    print(evaluation_file)


# ============================================================
# 2. Calculate ranking distribution
# ============================================================

if "relevant_rank" not in evaluation_df.columns:
    raise KeyError(
        "The evaluation dataframe does not contain relevant_rank."
    )

ranks = pd.to_numeric(
    evaluation_df["relevant_rank"],
    errors="coerce"
)

total_questions = len(evaluation_df)

rank_distribution_df = pd.DataFrame({
    "Ranking Position": [
        "Rank 1",
        "Rank 2",
        "Rank 3",
        "Not Found in Top 3"
    ],
    "Questions": [
        int((ranks == 1).sum()),
        int((ranks == 2).sum()),
        int((ranks == 3).sum()),
        int((~ranks.isin([1, 2, 3])).sum())
    ]
})

rank_distribution_df["Percentage"] = (
    rank_distribution_df["Questions"]
    / total_questions
    * 100
)

if rank_distribution_df["Questions"].sum() != total_questions:
    raise ValueError(
        "Ranking-distribution counts do not match the total questions."
    )

display(
    rank_distribution_df.style.format({
        "Questions": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)


# ============================================================
# 3. Generate visualization
# ============================================================

colors = [
    "#2E7D32",   # Rank 1
    "#4C78A8",   # Rank 2
    "#F2A541",   # Rank 3
    "#C94C4C"    # Not found
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    rank_distribution_df["Ranking Position"],
    rank_distribution_df["Questions"],
    color=colors,
    edgecolor="black",
    linewidth=0.7,
    width=0.68
)

ax.set_title(
    "Ranking Distribution of Expected Handbook Subsections",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Position of the Expected Subsection",
    fontsize=11
)

ax.set_ylabel(
    "Number of Questions",
    fontsize=11
)

maximum_count = rank_distribution_df["Questions"].max()

ax.set_ylim(0, maximum_count * 1.20)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

# Display count and percentage above every bar
for bar, count, percentage in zip(
    bars,
    rank_distribution_df["Questions"],
    rank_distribution_df["Percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + maximum_count * 0.025,
        f"{count}\n({percentage:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )


# ============================================================
# 4. Add Hit@3 summary
# ============================================================

hit_at_3_count = int(
    ranks.isin([1, 2, 3]).sum()
)

hit_at_3 = (
    hit_at_3_count / total_questions
)

ax.text(
    0.99,
    0.96,
    (
        f"Top-3 hits: {hit_at_3_count}/{total_questions}\n"
        f"Hit@3 = {hit_at_3:.4f}"
    ),
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#777777",
        alpha=0.9
    )
)

plt.tight_layout()


# ============================================================
# 5. Save figure and table
# ============================================================

drive_folders = [
    Path("/content/drive/MyDrive/FYP2"),
    Path("/content/gdrive/MyDrive/FYP2")
]

fyp2_folder = next(
    (
        folder for folder in drive_folders
        if folder.exists()
    ),
    None
)

if fyp2_folder is not None:
    output_dir = (
        fyp2_folder
        / "RAG_Evaluation_Figures"
    )
else:
    output_dir = Path(
        "/content/RAG_Evaluation_Figures"
    )

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

figure_path = (
    output_dir
    / "Figure_6.1.4.2_Ranking_Distribution_of_Expected_Subsections.png"
)

csv_path = (
    output_dir
    / "Table_6.1.4.2_Ranking_Distribution.csv"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

rank_distribution_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

plt.show()
plt.close(fig)

print("\nFigure saved to:")
print(figure_path)

print("\nTable saved to:")
print(csv_path)

In [ ]:
conversation_state_prompt = ChatPromptTemplate.from_template("""
Extract explicitly stated user information from the current message.

Existing conversation state:
{conversation_state}

Recent conversation:
{conversation_history}

Current message:
{user_input}

Rules:
- Extract the user's name only when the user clearly identifies themselves.
- Do not treat a doctor's name, family member's name or mentioned person's name
  as the user's name.
- Do not guess missing information.
- Keep the existing value when no new information is provided.
- Return valid JSON only.

Return:
{{
  "user_name": null,
  "preferred_language": null,
  "doctor_name": null,
  "appointment_date": null,
  "appointment_time": null,
  "symptoms": [],
  "duration": null,
  "severity": null
}}
""")

conversation_state_chain = (
    conversation_state_prompt
    | model
    | StrOutputParser()
)

In [ ]:
conversation_state = {
    "user_name": None,
    "preferred_language": None,
    "last_intent": None,
    "active_flow": None,
    "doctor_name": None,
    "appointment_date": None,
    "appointment_time": None,
    "symptoms": [],
    "duration": None,
    "severity": None,
    "history": []
}

## 11. Runtime Usage Note
This notebook is now the validated source for the revised RAG workflow.

The chatbot interface and deployed RAG runtime are intentionally **not updated yet**. After the notebook tests pass, the SQL-only slot branch can be exported to the runtime and integrated with the chatbot.
